# py-kits: one test per routine

Every routine is tested against an analytic or hand-derivable result, **never against files computed earlier**. All data are synthetic and small, except a few reads of grid coordinates, where the geometry of the real grid is the thing being tested.

How to read it: each test states the expected result in LaTeX, then runs it. Each check prints `PASS` or `FAIL` with the measured error against the tolerance. A routine that raises is recorded as `ERROR` and the notebook carries on. The last two cells give the full list of failures and check that every routine in the kits was covered.

A `FAIL` here means the routine disagrees with the mathematics or with its own docstring. The failures have deliberately been left visible and are not silenced.

In [ ]:
import ast, glob, importlib, os, subprocess, sys, tempfile, textwrap, time, types, warnings
import numpy as np, netCDF4
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import k1, erf
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "tests" else os.getcwd()
sys.path.insert(0, ROOT)
sys.dont_write_bytecode = True
TMP = tempfile.mkdtemp(prefix="pykits_")
RESULTS, COVERED = [], set()
_cur = [None]
class test:
    def __init__(self, *names): self.names = names
    def __enter__(self):
        COVERED.update(self.names); _cur[0] = self.names[0]; print("==", ", ".join(self.names)); return self
    def __exit__(self, et, ev, tb):
        if et is not None:
            RESULTS.append((self.names[0], "ERROR", f"{et.__name__}: {ev}", np.nan, np.nan)); print(f"   ERROR {et.__name__}: {ev}")
        return True
def ok(desc, err, tol):
    err = float(np.max(np.abs(err)))
    st = "PASS" if err <= tol else "FAIL"
    RESULTS.append((_cur[0], st, desc, err, tol)); print(f"   {st}  {desc}: {err:.3e} (tol {tol:.1e})")
def yes(desc, cond):
    st = "PASS" if bool(cond) else "FAIL"
    RESULTS.append((_cur[0], st, desc, np.nan, np.nan)); print(f"   {st}  {desc}")
def rerr(a, b): return np.max(np.abs(np.asarray(a) - b)) / max(np.max(np.abs(b)), 1e-300)
def ncfile(name, dims, **vars):
    p = os.path.join(TMP, name); os.makedirs(os.path.dirname(p), exist_ok=True)
    with netCDF4.Dataset(p, "w") as nc:
        for d, n in dims.items(): nc.createDimension(d, n)
        for v, (dn, a) in vars.items(): nc.createVariable(v, "f8", dn)[...] = a
    return p
A = 6371e3
def sph(nx, ny, latmax=np.pi/3):
    "Lat-lon C-grid on a sphere, with ORCA's period nx-2. T(j,i), U(j,i+1/2), V(j+1/2,i), F(j+1/2,i+1/2)."
    dl, dp = 2*np.pi/(nx-2), 2*latmax/ny
    lt = -latmax + (np.arange(ny)+.5)*dp; lv = lt + dp/2
    lo = (np.arange(nx)-.5)*dl; lu = lo + dl/2
    g = dict(nx=nx, ny=ny, lat_t=lt[:,None]*np.ones((ny,nx)), lat_v=lv[:,None]*np.ones((ny,nx)),
             lon_t=lo[None]*np.ones((ny,nx)), lon_u=lu[None]*np.ones((ny,nx)))
    g["e1t"] = g["e1u"] = A*np.cos(g["lat_t"])*dl; g["e2t"] = g["e2u"] = g["e2v"] = A*dp*np.ones((ny,nx))
    g["e1v"] = A*np.cos(g["lat_v"])*dl
    return g
def load_funcs(path, names, **glob_):
    "Run only the named functions of a script whose top level does real work."
    src = open(os.path.join(ROOT, path)).read(); ns = dict(glob_)
    for n in ast.parse(src).body:
        if isinstance(n, ast.FunctionDef) and n.name in names:
            exec(compile(ast.Module([n], []), path, "exec"), ns)
    return ns
from orcakit import orca, grid_c, filters, netcdf as onc, mpi_orca as mo
print("root", ROOT)

## 1. Differential operators on the sphere (`orcakit.grid_c`)

Tested on a periodic lat-lon C-grid of radius $a$, with $e_1=a\cos\varphi\,\Delta\lambda$ and $e_2=a\,\Delta\varphi$. Take

$$u=\cos\varphi\,\sin 2\lambda,\qquad v=\cos\varphi\,\sin\varphi\,\cos 3\lambda .$$

### `hDiv_Vgrid`: divergence of staggered velocities

$$\nabla\!\cdot\mathbf u=\frac{1}{a\cos\varphi}\Big[\partial_\lambda u+\partial_\varphi(v\cos\varphi)\Big]=\frac{2\cos\varphi\cos2\lambda+(\cos^3\varphi-2\cos\varphi\sin^2\varphi)\cos3\lambda}{a\cos\varphi}$$

A second-order scheme must divide the error by 4 when the grid is refined by 2, so the measured order $\log_2(E_N/E_{2N})$ must be about 2. The discrete form must also telescope: with $v=0$ on the first and last rows, $\sum \nabla\!\cdot\mathbf u\;e_1e_2=0$ to machine precision.

In [ ]:
def div_exact(g):
    p, l = g["lat_t"], g["lon_t"]
    return (2*np.cos(p)*np.cos(2*l) + (np.cos(p)**3 - 2*np.cos(p)*np.sin(p)**2)*np.cos(3*l))/(A*np.cos(p))
def uv_c(g):
    return np.cos(g["lat_t"])*np.sin(2*g["lon_u"]), np.cos(g["lat_v"])*np.sin(g["lat_v"])*np.cos(3*g["lon_t"])
with test("orcakit.grid_c.hDiv_Vgrid"):
    E = []
    for N in (180, 360):
        g = sph(N, N//2); u, v = uv_c(g)
        d = grid_c.hDiv_Vgrid(u, v, g["e1t"], g["e2t"], g["e1v"], g["e2u"], orca.halo)
        E.append(np.max(np.abs(d - div_exact(g))[1:-1]) )
    print("   max errors", E)
    ok("relative error, N=360", E[1]/np.abs(div_exact(g)).max(), 2e-3)
    ok("order of accuracy minus 2", np.log2(E[0]/E[1]) - 2, 0.2)
    v[0] = v[-1] = 0
    d = grid_c.hDiv_Vgrid(u, v, g["e1t"], g["e2t"], g["e1v"], g["e2u"], orca.halo)
    ok("discrete divergence theorem, sum/(sum|.|)", np.sum((d*g["e1t"]*g["e2t"])[:, 1:-1])/np.sum(np.abs(d*g["e1t"]*g["e2t"])[:, 1:-1]), 1e-13)

### `hDiv_Tgrid`: divergence of T-point velocities

Same analytic result, with $u,v$ given on T-points and averaged onto the faces first. Column 0 has a real western neighbour on a periodic grid, so it must be as accurate as every other column.

In [ ]:
with test("orcakit.grid_c.hDiv_Tgrid"):
    E = []
    for N in (180, 360):
        g = sph(N, N//2); p, l = g["lat_t"], g["lon_t"]
        u, v = np.cos(p)*np.sin(2*l), np.cos(p)*np.sin(p)*np.cos(3*l)
        d = grid_c.hDiv_Tgrid(u, v, g["e1t"], g["e2t"], g["e1v"], g["e2u"], orca.halo)
        E.append(np.max(np.abs(d - div_exact(g))[1:-1, 1:]))
    ok("relative error, N=360, excluding column 0", E[1]/np.abs(div_exact(g)).max(), 2e-3)
    ok("order of accuracy minus 2", np.log2(E[0]/E[1]) - 2, 0.2)
    ok("relative error on column 0", np.max(np.abs(d - div_exact(g))[1:-1, 0])/np.abs(div_exact(g)).max(), 2e-3)

### `vort_Vgrid`: relative vorticity at F-points

$$\zeta=\frac{1}{a\cos\varphi}\Big[\partial_\lambda v-\partial_\varphi(u\cos\varphi)\Big]=\frac{-3\cos\varphi\sin\varphi\sin3\lambda+2\cos\varphi\sin\varphi\sin2\lambda}{a\cos\varphi}$$

Evaluated at $F=(\varphi_{j+1/2},\lambda_{i+1/2})$. For solid-body rotation, $u=\Omega a\cos\varphi$ and $v=0$, the result must be $\zeta=2\Omega\sin\varphi$, the textbook value.

In [ ]:
with test("orcakit.grid_c.vort_Vgrid"):
    E = []
    for N in (180, 360):
        g = sph(N, N//2); u, v = uv_c(g); pf, lf = g["lat_v"], g["lon_u"]
        ex = (-3*np.sin(pf)*np.sin(3*lf) + 2*np.sin(pf)*np.sin(2*lf))/A
        z = grid_c.vort_Vgrid(u, v, g["e1v"], g["e2t"], g["e1u"], g["e2v"], orca.halo)
        E.append(np.max(np.abs(z - ex)[1:-2]))
    ok("relative error, N=360", E[1]/np.abs(ex).max(), 3e-3)
    ok("order of accuracy minus 2", np.log2(E[0]/E[1]) - 2, 0.2)
    Om = 1e-5; us = Om*A*np.cos(g["lat_t"])
    z = grid_c.vort_Vgrid(us, 0*us, g["e1v"], g["e2t"], g["e1u"], g["e2v"], orca.halo)
    ok("solid-body rotation, against 2 Omega sin(phi)", rerr(z[1:-2], (2*Om*np.sin(pf))[1:-2]), 2e-3)

### `fdiff_1D`, `bdiff_1D`: derivatives, and two identities they must satisfy

For $\psi=\cos\varphi\cos2\lambda$: $\;\partial_x\psi=-2\sin2\lambda/a$ and $\partial_y\psi=-\sin\varphi\cos2\lambda/a$.

Two discrete identities hold exactly, not just to second order:

$$\nabla\times\nabla\psi=0,\qquad \nabla\cdot\nabla^\perp\psi=0 .$$

Each is built from the forward or backward differences and checked at machine precision relative to the size of the individual terms.

In [ ]:
with test("orcakit.grid_c.fdiff_1D", "orcakit.grid_c.bdiff_1D"):
    g = sph(360, 180); DL = 2*np.pi/358; DP = 2*(np.pi/3)/180; p, l = g["lat_t"], g["lon_t"]; psi = np.cos(p)*np.cos(2*l)
    dx = grid_c.fdiff_1D(psi, g["e1u"], orca.halo, dir="lon")
    ok("fdiff lon at U-points", rerr(dx[1:-1], (-2*np.sin(2*g["lon_u"])/A)[1:-1]), 2e-3)
    dy = grid_c.fdiff_1D(psi, g["e2v"], orca.halo, dir="lat")
    ok("fdiff lat at V-points", rerr(dy[1:-2], (-np.sin(g["lat_v"])*np.cos(2*l)/A)[1:-2]), 2e-3)
    bx = grid_c.bdiff_1D(psi, g["e1u"], orca.halo, dir="lon")
    ok("bdiff lon at U(i-1/2)", rerr(bx[1:-1], (-2*np.sin(2*(l-DL/2))/A)[1:-1]), 2e-3)
    by = grid_c.bdiff_1D(psi, g["e2t"], orca.halo, dir="lat")
    ok("bdiff lat at V(j-1/2)", rerr(by[1:-1], (-np.sin(p-DP/2)*np.cos(2*l)/A)[1:-1]), 3e-3)
    z = grid_c.vort_Vgrid(dx, dy, g["e1v"], g["e2t"], g["e1u"], g["e2v"], orca.halo)
    ok("curl grad = 0, relative to |grad|/dx", np.abs(z[1:-2]).max()/(np.abs(dy).max()/g["e2t"].max()), 1e-12)
    pf = np.cos(g["lat_v"])*np.cos(2*g["lon_u"])
    uu = -grid_c.bdiff_1D(pf, g["e2u"], orca.halo, dir="lat"); vv = grid_c.bdiff_1D(pf, g["e1v"], orca.halo, dir="lon")
    d = grid_c.hDiv_Vgrid(uu, vv, g["e1t"], g["e2t"], g["e1v"], g["e2u"], orca.halo)
    ok("div curl = 0, relative to |u|/dx", np.abs(d[2:-2]).max()/(np.abs(uu).max()/g["e2t"].max()), 1e-12)

### `dxFu_1D`: zonal flux divergence

With $F_u=u\,e_{2u}$ the routine returns $\frac{1}{e_{1t}}\delta_iF_u\to e_{2u}\,\frac{1}{a\cos\varphi}\partial_\lambda u$, with no division by $e_{2t}$, as its docstring says ($dF_u/dx$). For $u=\cos\varphi\sin2\lambda$ this is $2e_{2u}\cos2\lambda/a$.

In [ ]:
with test("orcakit.grid_c.dxFu_1D"):
    g = sph(360, 180); u = np.cos(g["lat_t"])*np.sin(2*g["lon_u"])
    d = grid_c.dxFu_1D(u, g["e1t"], g["e2u"], orca.halo)
    ok("against 2 e2u cos(2 lambda)/a", rerr(d, 2*g["e2u"]*np.cos(2*g["lon_t"])/A), 2e-3)

### `interp_T2UV`, `interp_1D`, `interp_UV2T`: staggered averages

$\tfrac12(f_i+f_{i+1})$ is exact for fields linear in the index, up to the wrap. With weights it is $\frac{w_if_i+w_{i+1}f_{i+1}}{w_i+w_{i+1}}$, compared below with that formula evaluated directly. A constant field must stay constant under any weights.

In [ ]:
with test("orcakit.grid_c.interp_T2UV", "orcakit.grid_c.interp_1D", "orcakit.grid_c.interp_UV2T"):
    ny, nx = 8, 12; jj, ii = np.mgrid[:ny, :nx].astype(float); f = 3*jj + 2*ii + 1; w = 1 + (jj+ii) % 3
    for name, fn in (("T2UV", grid_c.interp_T2UV), ("1D", grid_c.interp_1D)):
        ok(f"{name} lon linear", rerr(fn(f, orca.halo, dir="lon")[:, :-1], (f + 1.0)[:, :-1]), 1e-14)
        ok(f"{name} lat linear", rerr(fn(f, orca.halo, dir="lat")[:-1], (f + 1.5)[:-1]), 1e-14)
        ok(f"{name} lon weighted, constant stays", rerr(fn(0*f+7, orca.halo, weights_in=w, dir="lon"), 7.), 1e-14)
        h = (w[:, :-1]*f[:, :-1] + w[:, 1:]*f[:, 1:])/(w[:, :-1]+w[:, 1:])
        ok(f"{name} lon weighted, hand value", rerr(fn(f, orca.halo, weights_in=w, dir="lon")[:, :-1], h), 1e-14)
        h = (w[:-1]*f[:-1] + w[1:]*f[1:])/(w[:-1]+w[1:])
        ok(f"{name} lat weighted, hand value", rerr(fn(f, orca.halo, weights_in=w, dir="lat")[:-1], h), 1e-14)
    ok("UV2T lon linear", rerr(grid_c.interp_UV2T(f, orca.halo, dir="lon")[:, 1:], (f - 1.0)[:, 1:]), 1e-14)
    ok("UV2T lat linear", rerr(grid_c.interp_UV2T(f, orca.halo, dir="lat")[1:], (f - 1.5)[1:]), 1e-14)
    ok("UV2T lon weighted, constant", rerr(grid_c.interp_UV2T(0*f+7, orca.halo, weights_in=w, dir="lon"), 7.), 1e-14)
    ok("UV2T lat weighted, constant", rerr(grid_c.interp_UV2T(0*f+7, orca.halo, weights_in=w, dir="lat"), 7.), 1e-14)

### `ke_compute`, `uv2t`

For uniform $u=U$, $v=V$ on interior T-points: `ke_compute` $=2(U^2+V^2)$, which is 4 times $\tfrac12(U^2+V^2)$, and `uv2t` $=2(U+V)$.

In [ ]:
with test("orcakit.grid_c.ke_compute", "orcakit.grid_c.uv2t"):
    u, v = 3*np.ones((6, 7)), 4*np.ones((6, 7))
    ok("ke_compute interior", grid_c.ke_compute(u, v)[1:-1, 1:-1] - 50, 1e-13)
    ok("uv2t interior", grid_c.uv2t(u, v)[1:-1, 1:-1] - 14, 1e-13)

## 2. Boundary exchange and the north fold (`orcakit.orca`)

### `halo`, `dehalo`: bookkeeping

With a label field $f_{j,i}=1000j+i$, each ghost cell's value says where it came from: the east ghost must hold $f_{j,2}$, the west ghost $f_{j,n_x-3}$ (the grid has period $n_x-2$), and `dehalo(halo(f))` must return $f$. A complex field must keep its imaginary part.

In [ ]:
with test("orcakit.orca.halo", "orcakit.orca.dehalo", "orcakit.orca.wrap_columns", "orcakit.orca.fold_columns"):
    ny, nx = 8, 12; f = 1000.*np.arange(ny)[:, None] + np.arange(nx)
    h = orca.halo(f, np.array([1, 0, 1]))
    ok("east ghost = column 2", h[:, -1] - f[:, 2], 0); ok("west ghost = column nx-3", h[:, 0] - f[:, nx-3], 0)
    for ha in ([1,2,1], [0,1,0], [2,0,0], [0,0,2], [3,3,3]):
        ha = np.array(ha); ok(f"dehalo(halo) with {list(ha)}", orca.dehalo(orca.halo(f, ha), ha) - f, 0)
    ok("isMeridional flips the north ghost only", orca.halo(f, np.array([0,1,0]), True)[-1] + orca.halo(f, np.array([0,1,0]))[-1], 0)
    for P, r, c in (("T", 4, 0), ("U", 4, -1), ("V", 5, 0), ("F", 5, -1)):
        ok(f"north ghost source, {P}", orca.halo(f, np.array([0,2,0]), point=P)[ny+1] - f[ny-r-1, (nx+c-np.arange(nx)) % (nx-2)], 0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ok("complex field keeps its imaginary part", orca.halo(1j*f, np.array([1,1,1])).imag.max() - f.max(), 0)

### `halo` against the geometry of the real grids

This is the one place the real grid is read, only `nav_lat` and `nav_lon`. A halo is correct if the ghost cell is **the point that physically lies there**. The mesh itself tells us its symmetry. For a T-pivot grid with two overlap columns:

$$x_{n_y-1,\,i}=x_{n_y-3,\,n_x-i},\qquad x_{j,\,i}=x_{j,\,i+n_x-2}.$$

So the next row and column continue as

$$x_{n_y,\,i}=x_{n_y-4,\,n_x-i},\qquad x_{j,\,n_x}=x_{j,\,2},\qquad x_{j,\,-1}=x_{j,\,n_x-3}.$$

First the symmetry is checked on the mesh, then the distance in km between each `halo` ghost and that true point, compared with the local grid spacing. A correct exchange gives a small fraction of one cell.

In [ ]:
from datakit import glorys04
def xyz(lat, lon):
    la, lo = np.deg2rad(lat), np.deg2rad(lon); return np.stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)])
la, lo = [np.asarray(a, float) for a in glorys04.mesh("nav_lat", "nav_lon")]; ny, nx = la.shape; X = xyz(la, lo); RE = 6371.
cell = RE*np.linalg.norm(X[:, ny-3, nx//4:3*nx//4]-X[:, ny-2, nx//4:3*nx//4], axis=0).mean()
with test("orcakit.orca.halo (geometry, GLORYS04 mesh)"):
    msh = netCDF4.Dataset(glorys04.MESH[0])
    for P in "tuvf":
        Xp = xyz(np.squeeze(msh["gphi"+P][...]).astype(float), np.squeeze(msh["glam"+P][...]).astype(float)); rr, cc = orca.FOLD[P.upper()]
        dm = RE*np.linalg.norm(Xp[:, ny-1]-Xp[:, ny-rr+1][:, orca.fold_columns(np.arange(nx), nx, P.upper())], axis=0)
        print(f"   {P}: last row against the fold rule: median {np.median(dm):.3f} km, 99th percentile {np.percentile(dm, 99):.3f} km, max {dm.max():.1f} km (at the pivot and the two poles)")
        ok(f"{P}: the mesh obeys the fold rule on its last row, 99th percentile, km", np.percentile(dm, 99), 0.5)
        h = np.stack([orca.halo(c, np.array([3, 3, 3]), point=P.upper()) for c in Xp]); jm = ny//2
        step = RE*np.linalg.norm(Xp[:, jm, 3]-Xp[:, jm, 2])
        ok(f"{P}: east ghost is the true point, cells", RE*np.linalg.norm(h[:, jm, 3+nx]-Xp[:, jm, 2])/step, 1e-6)
        ok(f"{P}: west ghost is the true point, cells", RE*np.linalg.norm(h[:, jm, 2]-Xp[:, jm, nx-3])/step, 1e-6)
        sp = np.median(RE*np.linalg.norm(Xp[:, ny-2, nx//4:3*nx//4]-Xp[:, ny-3, nx//4:3*nx//4], axis=0))
        gh = np.median(RE*np.linalg.norm(h[:, ny:ny+3, nx//4:3*nx//4]-h[:, ny-1:ny+2, nx//4:3*nx//4], axis=0), axis=1)
        ok(f"{P}: ghost rows continue the grid spacing, relative", gh/sp - 1, 0.05)
    tm0 = np.squeeze(msh["tmask"][...])[0].astype(float)
    ok("create_umask_from_tmask = NEMO umask", orca.create_umask_from_tmask(tm0) - np.squeeze(msh["umask"][...])[0], 0)
    ok("create_vmask_from_tmask = NEMO vmask", orca.create_vmask_from_tmask(tm0) - np.squeeze(msh["vmask"][...])[0], 0)


### North-fold sign of a vector component

Across the fold both grid directions reverse, $\hat{\mathbf i}\to-\hat{\mathbf i}$ and $\hat{\mathbf j}\to-\hat{\mathbf j}$. So **both** $u$ and $v$ change sign, which is what NEMO does with `lbc_lnk(..., 'U', -1)` and `lbc_lnk(..., 'V', -1)`.

Project a constant Cartesian vector on the local grid directions, $u=\mathbf V\cdot\hat{\mathbf i}$ and $v=\mathbf V\cdot\hat{\mathbf j}$. That field is smooth, so the jump between the last real row and the ghost row must be small compared with $|u|$. Comparing `isMeridional=False` and `True` shows which sign `halo` must be told to use for $u$.

In [ ]:
with test("orcakit.orca.halo (fold sign)"):
    W = np.array([0.3, -0.5, 0.2])[:, None, None]
    ei = np.gradient(X, axis=2); ei /= np.linalg.norm(ei, axis=0); ej = np.gradient(X, axis=1); ej /= np.linalg.norm(ej, axis=0)
    u, v = (W*ei).sum(0), (W*ej).sum(0)
    sel = slice(nx//8, 3*nx//8)
    jump = lambda f, m, P: np.median(np.abs(orca.halo(f, np.array([0,1,0]), m, P)[ny, sel] - f[ny-1, sel]))/np.median(np.abs(f[ny-1, sel]))
    print(f"   u: jump {jump(u, False, "U"):.2f} with +1, {jump(u, True, "U"):.2f} with -1;  v: jump {jump(v, False, "V"):.2f} with +1, {jump(v, True, "V"):.2f} with -1")
    yes("-1 gives the smaller jump for u", jump(u, True, "U") < jump(u, False, "U"))
    yes("-1 gives the smaller jump for v", jump(v, True, "V") < jump(v, False, "V"))
    src = open(os.path.join(ROOT, "orcakit/grid_c.py")).read()
    yes("vort_Vgrid exchanges U with the -1 fold sign", 'exchange(Fu, np.array([0, 1, 0]), True, "U")' in src)

### `smooth_2D`: Gaussian transfer function

The kernel $G_\sigma$ multiplies a wave of wavenumber $k$ (radians per grid point) by

$$\hat G(k)=e^{-k^2\sigma^2/2}.$$

For $f=\cos(kx)$ with $k=2\pi m/(n_x-2)$, away from the fold, the output must be $\hat G f$. A constant must stay constant, and an applied mask must multiply the result.

In [ ]:
with test("orcakit.orca.smooth_2D"):
    ny, nx = 90, 240; m, s = 6, 4.; k = 2*np.pi*m/(nx-2); x = np.arange(nx)
    f = np.cos(k*x)[None]*np.ones((ny, nx))
    ok("plane wave against exp(-k^2 sigma^2/2)", rerr(orca.smooth_2D(f, s)[20:-20], (np.exp(-k**2*s**2/2)*f)[20:-20]), 2e-3)
    ok("constant preserved", orca.smooth_2D(0*f+5, s) - 5, 1e-12)
    s2 = 2.5; ok("non-integer sigma = 2.5 honoured", rerr(orca.smooth_2D(f, s2)[20:-20], (np.exp(-k**2*s2**2/2)*f)[20:-20]), 2e-3)
    mk = np.ones((ny, nx)); mk[:10] = 0
    ok("mask applied", orca.smooth_2D(f, s, mask=mk)[:10], 0)

### `find_nearest_LatLon`, `find_nearest_grid`, `load_or_make_map`, `downsample_field`

On a regular grid the nearest point is known exactly: $j=\mathrm{round}\big((\varphi-\varphi_0)/\Delta\big)$ and likewise for $i$. Longitude is periodic, so $\lambda=179.9^\circ$ on a grid running from $-180^\circ$ to $179^\circ$ is nearest to $-180^\circ$, at $0.1^\circ$, not to $179^\circ$, at $0.9^\circ$.

`downsample_field` must return $f(j^\star,i^\star)$ at each masked point, keep the leading axes, and keep complex values. `load_or_make_map` must return the same indices when it computes them and when it reloads them.

In [ ]:
with test("orcakit.orca._unit_sphere", "orcakit.orca.find_nearest_LatLon", "orcakit.orca.find_nearest_grid", "orcakit.orca.load_or_make_map", "orcakit.orca.downsample_field"):
    lat = np.arange(-60., 61)[:, None]*np.ones((121, 360)); lon = np.arange(-180., 180)[None]*np.ones((121, 360))
    ja, ia = orca.find_nearest_LatLon(lat, lon, 10.3, 20.2); ok("regular grid", [ja-70, ia-200], 0)
    ja, ia = orca.find_nearest_LatLon(lat, lon, 0., 179.9); ok("longitude wrap: index of -180", ia - 0, 0)
    latc = np.array([[-10.2, 3.1], [20.4, 40.6]]); lonc = np.array([[-20.3, 50.2], [100.1, 120.4]])
    jm, im = orca.find_nearest_grid(latc, lonc, lat, lon)
    ok("grid lat", jm - (np.round(latc)+60), 0); ok("grid lon", im - (np.round(lonc)+180), 0)
    mk = np.array([[True, False], [True, True]]); jm2, im2 = orca.find_nearest_grid(latc, lonc, lat, lon, mask=mk)
    ok("masked points stay -999", jm2[0, 1] + 999, 0)
    p = os.path.join(TMP, "map.nc"); a = orca.load_or_make_map(p, latc, lonc, lat, lon); b = orca.load_or_make_map(p, latc, lonc, lat, lon)
    ok("map made", a[0]-jm, 0); ok("map reloaded", b[1]-im, 0)
    f = 1000.*np.arange(121)[:, None] + np.arange(360)
    ds = orca.downsample_field(f, jm, im, mk); ok("sampled values", ds[mk] - (1000.*jm + im)[mk], 0); ok("off-mask zero", ds[0, 1], 0)
    d3 = orca.downsample_field(np.stack([f, 2*f]) * (1+1j), jm, im, mk)
    ok("leading axis and complex", d3[1][mk] - 2*(1+1j)*(1000.*jm + im)[mk], 0)

### `create_tmask_from_top_and_bottom`, `create_umask_from_tmask`, `create_vmask_from_tmask`

NEMO definitions: $t_{k,j,i}=1$ for $\text{top}\le k\le\text{bot}$ (1-based $k$), else 0; $u_{k,j,i}=t_{k,j,i}\,t_{k,j,i+1}$ and $v_{k,j,i}=t_{k,j,i}\,t_{k,j+1,i}$.

In [ ]:
with test("orcakit.orca.create_tmask_from_top_and_bottom", "orcakit.orca.create_umask_from_tmask", "orcakit.orca.create_vmask_from_tmask"):
    top = np.array([[1, 1, 2], [1, 0, 1], [1, 1, 1]]); bot = np.array([[3, 2, 3], [1, 0, 2], [2, 2, 2]])
    t = orca.create_tmask_from_top_and_bottom(top, bot, 3, 3, 3)
    ok("column (0,2) = 0,1,1", t[:, 0, 2] - [0, 1, 1], 0); ok("land column (1,1) with top=0", t[:, 1, 1], 0); ok("total wet cells", t.sum() - 16, 0)
    t = (np.random.default_rng(1).random((3, 8, 10)) > .3).astype(float)
    ok("umask = t(i) t(i+1), interior", orca.create_umask_from_tmask(t)[:, :, :-1] - t[:, :, :-1]*t[:, :, 1:], 0)
    ok("vmask = t(j) t(j+1), interior", orca.create_vmask_from_tmask(t)[:, :-1] - t[:, :-1]*t[:, 1:], 0)

### `e3uv_from_partial_steps`

$e_{3u}=\min(e_{3t}^{i},e_{3t}^{i+1})$ and $e_{3v}=\min(e_{3t}^{j},e_{3t}^{j+1})$, with $e_{3t}=e_{3t}^0$ above the bottom cell and $e_{3t}^{ps}$ in it. On a small column set this is computed by hand: reference thickness $10,20,30$, column $A$ with 3 levels and a bottom cell of 12, column $B$ with 2 levels and a bottom cell of 5.

In [ ]:
with test("orcakit.orca.e3uv_from_partial_steps"):
    e0 = np.array([10., 20., 30.]); ps = np.array([[12., 5., 12., 12.]]*4)
    t = np.ones((3, 4, 4), bool); t[2, :, 1] = False
    e3u, e3v = orca.e3uv_from_partial_steps(e0, ps, t, t, t)
    ok("U between A (10,20,12) and B (10,5,dry) = min", e3u[:, 1, 0] - [10, 5, 0], 1e-12)
    ok("V inside A", e3v[:, 1, 0] - [10, 20, 12], 1e-12)

### `split_barotropic`

For $f(z)=B+c(z)$ with $\int c\,e_3\,dz=0$, the split must return exactly $B$ and $c$. Take $e_3=(1,2,1)$ and $c=(1,-1,1)$, for which $\sum c\,e_3=0$. Without weights the mean over wet levels is used instead. Land columns must stay 0.

In [ ]:
with test("orcakit.orca.split_barotropic"):
    B = 2.5; c = np.array([1., -1., 1.])[:, None, None]; e3 = np.array([1., 2., 1.])[:, None, None]*np.ones((3, 2, 2)); mk = np.ones((3, 2, 2)); mk[:, 0, 0] = 0
    f = (B + c)*np.ones((3, 2, 2)); bt, bc = orca.split_barotropic(f, mk, e3=e3)
    ok("barotropic", bt[1, 1] - B, 1e-13); ok("baroclinic", bc[:, 1, 1] - c[:, 0, 0], 1e-13); ok("land", [bt[0, 0], *bc[:, 0, 0]], 0)
    bt2, _ = orca.split_barotropic(f, mk); ok("unweighted mean", bt2[1, 1] - (B + 1/3), 1e-13)
    btt, _ = orca.split_barotropic(np.stack([f, 2*f]), mk, e3=e3); ok("leading time axis", btt[1, 1, 1] - 2*B, 1e-13)

### `assign_to_coarse_grid`, `bin_to_coarse`

Fine points at known distances must go to the nearest wet coarse cell, across the $\pm180^\circ$ seam. The bin average is

$$\bar f_c=\frac{\sum_{p\in c}w_pf_p}{\sum_{p\in c}w_p}.$$

Land cells must be avoided, and a constant must stay constant.

In [ ]:
with test("orcakit.orca.assign_to_coarse_grid", "orcakit.orca.bin_to_coarse"):
    lc = np.array([[0., 0.]]); oc = np.array([[-179., 10.]]); mk = np.array([[True, True]])
    idx = orca.assign_to_coarse_grid(np.array([[0., 0., 0.]]), np.array([[179.5, 9., 11.]]), lc, oc, mk)
    ok("across the seam, then nearest", idx[0] - [0, 1, 1], 0)
    idx2 = orca.assign_to_coarse_grid(np.array([[0.]]), np.array([[9.]]), lc, oc, np.array([[True, False]]))
    ok("land cell avoided", idx2 - 0, 0)
    f = np.array([[1., 2., 4.]]); w = np.array([[1., 1., 3.]]); avg, cnt = orca.bin_to_coarse(f, w, idx, 3)
    ok("weighted average", avg - [1, 14/4, 0], 1e-14); ok("counts", cnt - [1, 4, 0], 0)
    a3, c3 = orca.bin_to_coarse(np.stack([f, 2*f]), np.stack([w, w]), idx, 3); ok("3D", a3[1] - [2, 7, 0], 1e-14)

### `coastal_taper`

For one pass and a straight coast, where the wet half-plane starts at index 0, the discrete Gaussian of a step gives about

$$T(d)\approx\frac12\Big[1+\mathrm{erf}\Big(\frac{d+\tfrac12}{\sqrt2\,\sigma}\Big)\Big].$$

Land stays 0, and an all-wet mask stays 1.

In [ ]:
with test("orcakit.orca.coastal_taper"):
    mk = np.zeros((60, 200)); mk[:, 100:] = 1; s = 6.
    t = orca.coastal_taper(mk, s, 1)[30, 100:130]; d = np.arange(30)
    ok("one pass against the erf profile", t - .5*(1+erf((d+.5)/(np.sqrt(2)*s))), 2e-3)
    ok("land stays 0", orca.coastal_taper(mk, s, 3)[:, :100], 0)
    ok("all-wet stays 1", orca.coastal_taper(np.ones((20, 20)), s, 3) - 1, 1e-5)

## 3. Filters and files (`orcakit.filters`, `orcakit.netcdf`)

### `masked_smooth`, `scale_split`

The ratio $\overline{fm}/\overline{m}$ must reproduce a **constant exactly, even next to land**, which is the point of the method. Far from land a wave must be attenuated by $e^{-k^2\sigma^2/2}$. Also $f=\text{large}+\text{small}$ exactly on wet points, and land stays 0.

In [ ]:
from functools import partial
with test("orcakit.filters.masked_smooth", "orcakit.filters.scale_split"):
    ny, nx = 90, 240; mk = np.ones((ny, nx)); mk[30:50, 60:100] = 0; sm = partial(orca.smooth_2D, mask=None)
    ok("constant stays exact next to land", (filters.masked_smooth(7*mk, mk, 4., sm) - 7)[mk > 0], 1e-11)
    ok("land stays 0", filters.masked_smooth(7*mk, mk, 4., sm)[mk == 0], 0)
    k = 2*np.pi*6/(nx-2); f = np.cos(k*np.arange(nx))[None]*np.ones((ny, nx))
    ok("wave far from land", rerr(filters.masked_smooth(f, np.ones_like(f), 4., sm)[20:-20], (np.exp(-8*k*k)*f)[20:-20]), 2e-3)
    ny7, nx7 = 60, 120; one = np.ones((ny7, nx7)); flip = partial(orca.smooth_2D, mask=None, isMeridional=True); keep = partial(orca.smooth_2D, mask=None)
    ok("a mask of ones gives den = 1 at the fold", filters.masked_smooth(one, one, 4., keep, keep) - 1, 1e-11)
    ok("with smooth_mask, a flipped field leaves den alone", np.abs(filters.masked_smooth(one, one, 4., flip, keep)[-1]).max() - (orca.smooth_2D(one, 4., isMeridional=True)[-1]).max(), 1e-11)
    L, S = filters.scale_split(f, mk, 4., sm); ok("large + small = field on wet", ((L+S)-f)[mk > 0], 1e-13); ok("land", S[mk == 0], 0)

### `nc_copy`, `write_fields`, `write_modes`

A write followed by a read must return the same numbers. NaN and masked values go to `fill_value`, and complex modes are split into `_real_r` and `_imag_r`.

In [ ]:
with test("orcakit.netcdf.nc_copy", "orcakit.netcdf.write_fields", "orcakit.netcdf.write_modes"):
    g = ncfile("grid.nc", dict(z=2, y=3, x=4), a=(("y", "x"), np.arange(12.).reshape(3, 4)), b=(("y", "x"), np.zeros((3, 4))))
    d = onc.nc_copy(g, os.path.join(TMP, "c.nc"), atts=True, dims=True, vars=True, exclude=["b"]);
    ok("variable copied", d["a"][:].data - np.arange(12.).reshape(3, 4), 0); yes("excluded variable left out", "b" not in d.variables); d.close()
    f3 = np.arange(24.).reshape(2, 3, 4); f3n = f3.copy(); f3n[0, 0, 0] = np.nan; f2 = np.ma.array(np.ones((3, 4)), mask=np.eye(3, 4, dtype=bool))
    out = os.path.join(TMP, "w.nc"); onc.write_fields(g, out, dict(p=f3n, q=f2))
    with netCDF4.Dataset(out) as nc:
        nc.set_auto_mask(False); ok("3D field", nc["p"][:][0, 1:] - f3[0, 1:], 1e-5); ok("NaN to fill", nc["p"][:][0, 0, 0], 0); ok("masked to fill", nc["q"][:][0, 0], 0); yes("dims follow the rank", nc["q"].dimensions == ("y", "x"))
    m = (1+2j)*np.ones((2, 2, 3, 4)); out = os.path.join(TMP, "m.nc"); onc.write_modes(out, dict(uavg=f3), dict(umode=m), np.array([.1, .2]))
    with netCDF4.Dataset(out) as nc:
        ok("mean", nc["uavg"][:] - f3, 0); ok("real", nc["umode_real_r"][:] - 1, 0); ok("imag", nc["umode_imag_r"][:] - 2, 0); ok("omega", nc["omega"][:] - [.1, .2], 0)

## 4. Flow and comparison diagnostics (`diagkit`)

### `flow.*`: strain, vorticity, Okubo-Weiss

For a linear flow $u=ax+by$, $v=cx+dy$ the derivatives are exact on any grid:

$$s_n=a-d,\quad s_s=c+b,\quad\zeta=c-b,\quad W=s_n^2+s_s^2-\zeta^2 .$$

Solid-body rotation has $W=-4\Omega^2$ and pure strain has $W>0$.

In [ ]:
from diagkit import flow
with test("diagkit.flow.strain_normal", "diagkit.flow.strain_shear", "diagkit.flow.vorticity", "diagkit.flow.okubo_weiss", "diagkit.flow._t_from_f"):
    ny, nx = 20, 30; dx, dy = 2., 3.; jj, ii = np.mgrid[:ny, :nx].astype(float); xu, yu = (ii+.5)*dx, jj*dy; xv, yv = ii*dx, (jj+.5)*dy
    a, b, c, d = .3, -.7, 1.1, .2; u = a*xu + b*yu; v = c*xv + d*yv; I = (slice(2, -2), slice(2, -2))
    ok("s_n", flow.strain_normal(u, v, dx, dy)[I] - (a-d), 1e-12); ok("s_s", flow.strain_shear(u, v, dx, dy)[I] - (c+b), 1e-12)
    ok("vorticity", flow.vorticity(u, v, dx, dy)[I] - (c-b), 1e-12)
    ok("Okubo-Weiss", flow.okubo_weiss(u, v, dx, dy)[I] - ((a-d)**2+(c+b)**2-(c-b)**2), 1e-11)
    ok("_t_from_f of a constant", flow._t_from_f(np.ones((5, 5)))[1:-1, 1:-1] - 1, 0)
    Om = .5; ok("rotation W = -4 Omega^2", flow.okubo_weiss(-Om*yu, Om*xv, dx, dy)[I] + 4*Om*Om, 1e-12)

### `metrics.*`

- $\mathrm{rms}(c)=|c|$; $\mathrm{rmse}(a+c,a)=|c|$.
- Pattern correlation: $\rho(f,\alpha f)=\mathrm{sign}\,\alpha$ and $\rho(\sin,\cos)=0$ over a full period.
- Gaussian relative entropy, the Kullback-Leibler divergence of two normals:
$$\mathrm{GRE}=\tfrac12\Big[\frac{(\mu_e-\mu_o)^2}{\sigma_e^2}+\frac{\sigma_o^2}{\sigma_e^2}-1-\ln\frac{\sigma_o^2}{\sigma_e^2}\Big],$$
which is 0 for identical distributions.

In [ ]:
from diagkit import metrics
with test("diagkit.metrics._wmean", "diagkit.metrics.rms", "diagkit.metrics.rmse", "diagkit.metrics.pattern_correlation", "diagkit.metrics.gaussian_relative_entropy", "diagkit.metrics.averaged_gre"):
    w = np.array([[1., 3.]]); ok("_wmean", metrics._wmean(np.array([[1., 5.]]), w) - 4, 0)
    ok("rms of a constant", metrics.rms(-2*np.ones((3, 4)), np.ones((3, 4))) - 2, 1e-14)
    f = np.sin(np.linspace(0, 2*np.pi, 64, endpoint=False))[None]; W = np.ones_like(f)
    ok("rmse", metrics.rmse(f+3, f, W) - 3, 1e-14)
    ok("corr(f, -2f) = -1", metrics.pattern_correlation(-2*f, f, W) + 1, 1e-14)
    ok("corr(sin, cos) = 0", metrics.pattern_correlation(f, np.cos(np.linspace(0, 2*np.pi, 64, endpoint=False))[None], W), 1e-14)
    ok("GRE identical", metrics.gaussian_relative_entropy(1., 1., 2., 2.), 1e-15)
    ok("GRE hand value", metrics.gaussian_relative_entropy(1., 0., 2., 1.) - .5*(.5 + .5 - 1 - np.log(.5)), 1e-14)
    ok("averaged GRE uniform", metrics.averaged_gre(W, 0*W, 2*W, W, W) - .5*(.5 + .5 - 1 - np.log(.5)), 1e-14)

### `spectra.*`: Parseval

For a square domain, with one-sided rings of width $\Delta k$:

$$\sum_p E(k_p)\,\Delta k=\langle f^2\rangle,$$

because `iso_spec` is normalised so the spectrum integrates to the variance. For $f=\cos(k_0x)$ all energy must fall in the ring holding $k_0$. In 3D, with `init_3Dwvnb`, the same Parseval relation must hold. For `temporal_psd`, a sine of amplitude $A$ has $\int\mathrm{PSD}\,df=A^2/2$ with a peak at $f_0$.

In [ ]:
from diagkit import spectra
with test("diagkit.spectra.init_2Dwvnb", "diagkit.spectra.iso_spec"):
    n = 64; L = 2*np.pi; dx = L/n; x = np.arange(n)*dx; f = np.cos(5*x)[None]*np.ones((n, n))
    wv, kr, efc = spectra.init_2Dwvnb(n, n, dx, dx); ok("first ring at 2 pi / L", kr[0] - 1, 1e-12); ok("wavenumber count", wv.size - n*n, 0)
    fh = np.abs(np.fft.fft2(f))**2; E = spectra.iso_spec(fh, wv, kr, efc)
    ok("Parseval", E.sum()*(kr[1]-kr[0]) - (f**2).mean(), 1e-12); ok("energy in the k=5 ring only", np.delete(E, 4), 1e-20)
    r = np.random.default_rng(0).standard_normal((n, n)); r -= r.mean(); F = np.fft.fft2(r); F[np.abs(wv.reshape(n, n)) > kr[-1]] = 0; r2 = np.fft.ifft2(F).real
    ok("Parseval, random field", rerr(spectra.iso_spec(np.abs(np.fft.fft2(r2))**2, wv, kr, efc).sum(), (r2**2).mean()), 1e-10)
with test("diagkit.spectra.init_3Dwvnb"):
    n = 8; dx = 2*np.pi/n; x = np.arange(n)*dx; f = np.cos(2*x)[None, None]*np.ones((n, n, n))
    wv, kr, efc = spectra.init_3Dwvnb(n, n, n, dx, dx, dx); ok("wavenumber count", wv.size - n**3, 0)
    E = spectra.iso_spec(np.abs(np.fft.fftn(f))**2, wv, kr, efc)
    ok("3D Parseval", E.sum()*(kr[1]-kr[0]) - (f**2).mean(), 1e-10)
with test("diagkit.spectra.temporal_psd"):
    nt = 400; fs = 4.; f0 = 0.5; t = np.arange(nt)/fs; q = (3*np.sin(2*np.pi*f0*t))[:, None, None]*np.ones((nt, 4, 4))
    fr, p = spectra.temporal_psd(q, fs); ok("integral = A^2/2", (p[:, 0, 0].sum()*(fr[1]-fr[0])) - 4.5, 1e-10); ok("peak", fr[np.argmax(p[:, 0, 0])] - f0, 1e-12)
    fr, pw = spectra.temporal_psd(q, fs, np.ones((4, 4))/16); ok("weighted equals one point", pw - p[:, 0, 0], 1e-11)

### `stats.*`

$\mathrm{KE}=\tfrac12(U^2+V^2)$ on interior points; fluctuations have zero mean over each of the two periods; the Hovmöller of a uniform field is that value; and the spectrum of $u=\cos(k_0x)$ integrates to $\langle u^2\rangle/2$, the mean kinetic energy.

In [ ]:
from diagkit import stats
with test("diagkit.stats.fluctuations", "diagkit.stats.kinetic_energy", "diagkit.stats.hovmoller", "diagkit.stats.ke_spectrum"):
    q = np.arange(10.)[:, None, None]*np.ones((10, 3, 3)); fl = stats.fluctuations(q, 4)
    ok("spin-up anomalies", fl[:4].mean(0), 1e-14); ok("rest", fl[4:].mean(0), 1e-14); ok("value", fl[0, 0, 0] + 1.5, 1e-14)
    ok("KE interior", stats.kinetic_energy(3*np.ones((5, 5)), 4*np.ones((5, 5)))[1:-1, 1:-1] - 12.5, 1e-13)
    ok("Hovmoller", stats.hovmoller(q)[3] - 3, 0)
    n = 64; dx = 2*np.pi/n; u = np.cos(5*np.arange(n)*dx)[None]*np.ones((n, n)); k, s = stats.ke_spectrum(u, 0*u, dx, dx)
    ok("KE spectrum integrates to the mean KE", s.sum()*(k[1]-k[0]) - .25, 1e-12)

### `conservation.*`

`wet_mask_3d` must have $\sum_k$ mask $=$ `bottom_level`. The volume mean of a field that is $s_1$ in the upper layer and $s_2$ in the lower one is $\frac{s_1V_1+s_2V_2}{V_1+V_2}$, and the returned layer volume is $\sum e_3A$.

In [ ]:
from diagkit import conservation as cons
with test("diagkit.conservation.wet_mask_3d", "diagkit.conservation.volume_weighted_mean"):
    bl = np.array([[2, 1], [0, 2]]); m = cons.wet_mask_3d(bl, 3); ok("sum over k = bottom_level", m.sum(0) - bl, 0)
    e3 = np.array([1., 3., 5.])[None, :, None, None]*np.ones((2, 3, 2, 2)); ar = np.array([[1., 2.], [9., 4.]])
    so = np.array([10., 20., 99.])[None, :, None, None]*np.ones((2, 3, 2, 2))
    ms, vol = cons.volume_weighted_mean(so, e3, m, ar)
    V1 = 1*(1+2+4); V2 = 3*(1+4)
    ok("mean", ms - (10*V1+20*V2)/(V1+V2), 1e-13); ok("volume", vol - (V1+V2), 1e-13)
    ms2, _ = cons.volume_weighted_mean(so, e3, m, ar, 1, 2); ok("one layer", ms2 - 20, 1e-13)

### `fancy_plots.*`

`latlon_box` on a regular grid has a known answer, with sides snapped to even. `seam_mask` must find no seam on a smooth grid and must find one at an artificial $360^\circ$ jump. Outlines trace the box perimeter. The locator is checked only for running, since a plot has no analytic result.

In [ ]:
from diagkit import fancy_plots as fp
with test("diagkit.fancy_plots.latlon_box", "diagkit.fancy_plots.zone_boxes", "diagkit.fancy_plots.seam_mask", "diagkit.fancy_plots.zone_outline_lonlat", "diagkit.fancy_plots.plot_zone_locator"):
    lat = np.arange(-89.5, 90)[:, None]*np.ones((180, 360)); lon = np.arange(-179.5, 180)[None]*np.ones((180, 360))
    box = fp.latlon_box(lat, lon, (0, 10), (20, 30)); ok("box", np.array(box) - [90, 100, 200, 210], 0)
    z = fp.zone_boxes(lat, lon); yes("5 zones", len(z) == 5)
    lon2 = lon.copy(); yes("no seam on a smooth grid inside", not fp.seam_mask(lon2[20:-20, 20:-20], lat[20:-20, 20:-20], 0).any())
    lon3 = (lon + 100 + 180) % 360 - 180; yes("seam found at the jump", fp.seam_mask(lon3, lat, 0).any())
    o = fp.zone_outline_lonlat("GulfStream", lon, lat, z); yes("four sides", len(o) == 4); ok("south side latitude", o[0][1] - lat[z["GulfStream"][0], 0], 0)
    yes("ACC is two circles", len(fp.zone_outline_lonlat("ACC", lon, lat, z)) == 2)
    ax = fp.plot_zone_locator("Nino3", lon, lat, np.zeros_like(lat, bool), z); yes("locator returns axes", ax is not None); plt.close("all")

## 5. Noise building blocks (`noisekit`)

### `noise.uv_from_ssh_dA`: geostrophic velocity

$$U_g=-\frac{g}{f}\,\partial_y\eta,\qquad V_g=+\frac{g}{f}\,\partial_x\eta .$$

For a planar $\eta=\alpha y+\beta x$ with uniform metrics and constant $f$ these are exact constants: $U_g=-g\alpha/f$ and $V_g=g\beta/f$. Testing $\alpha$ and $\beta$ one at a time separates the two components.

In [ ]:
from noisekit import noise
with test("orcakit.grid_c.geostrophic_uv"):
    dx, dy, f0, al, be = 1e4, 2e4, 1e-4, 1e-6, 2e-6; ny_, nx_ = 30, 40; O = np.ones((ny_, nx_)); lat0 = np.degrees(np.arcsin(f0/(2*7.292116e-5)))
    jj, ii = np.mgrid[:ny_, :nx_]; W = O.astype(bool); kw = dict(umask=W, vmask=W, e1t=dx*O, e2t=dy*O, e1v=dx*O, e2u=dy*O, gphiu=lat0*O, gphiv=lat0*O)
    Ug, Vg = grid_c.geostrophic_uv(al*jj*dy, W, exchange=orca.halo, **kw)
    ok("Ug from a northward slope", Ug[2:-5, 2:-3] + 9.80665*al/f0, 1e-9); ok("no Vg from it", Vg[2:-5, 2:-3], 1e-9)
    Ug, Vg = grid_c.geostrophic_uv(be*ii*dx, W, exchange=orca.halo, **kw)
    ok("Vg from an eastward slope", Vg[2:-5, 2:-3] - 9.80665*be/f0, 1e-9); ok("no Ug from it", Ug[2:-5, 2:-3], 1e-9)
    yes("row 0 of Ug is NaN", np.isnan(Ug[0]).all())
    Ug, Vg = grid_c.geostrophic_uv(al*jj*dy, W, exchange=orca.halo, **{**kw, "gphiu": 2.*O, "gphiv": 2.*O}); yes("NaN below min_lat", np.isnan(Ug).all() and np.isnan(Vg).all())
    Ld = W.copy(); Ld[15, 20] = False; s_ = al*jj*dy; s_ = np.where(Ld, s_, np.nan); um_ = Ld & np.roll(Ld, -1, 1); vm_ = Ld & np.roll(Ld, -1, 0)
    Ug, Vg = grid_c.geostrophic_uv(s_, Ld, exchange=orca.halo, **{**kw, "umask": um_, "vmask": vm_}); yes("land does not spread NaN to wet points", np.isfinite(Ug[10:20, 10:30][um_[10:20, 10:30]]).all())
    yes("land is NaN", np.isnan(Ug[15, 20]) and np.isnan(Ug[15, 19]) and np.isnan(Vg[15, 20]))
with test("noisekit.noise.uv_from_ssh_dA"):
    ny, nx = 20, 30; dx, dy = 1e4, 2e4; jj, ii = np.mgrid[:ny, :nx].astype(float); f0 = 1e-4; g = 9.806
    O = np.ones((ny, nx)); al, be = 1e-6, 3e-6
    Ug, Vg = noise.uv_from_ssh_dA(al*jj*dy, O*dx, O*dy, f0*O, dx*O, dy*O)
    ok("eta = alpha y: U = -g alpha/f", rerr(Ug, -g*al/f0), 1e-10); ok("eta = alpha y: V = 0", Vg/(g*al/f0), 1e-10)
    Ug, Vg = noise.uv_from_ssh_dA(be*ii*dx, O*dx, O*dy, f0*O, dx*O, dy*O)
    ok("eta = beta x: V = +g beta/f", rerr(Vg, g*be/f0), 1e-10); ok("eta = beta x: U = 0", Ug/(g*be/f0), 1e-10)

### Primitives: `lowpass`, `highpass`, `deseason`, `time_lowpass`, `coarse_grain`, `running_stat`

Each takes arrays and returns arrays, so a pipeline is only a wrapper of them.


In [ ]:
with test("orcakit.filters.gaussian", "orcakit.filters.lowpass", "orcakit.filters.highpass"):
    from orcakit import filters as FL
    ny_, nx_ = 60, 80; jj, ii = np.mgrid[:ny_, :nx_]; rs = np.random.default_rng(0); f_ = rs.normal(size=(ny_, nx_)); m_ = np.ones((ny_, nx_), bool); m_[20:30, 30:45] = False
    lo = FL.lowpass(f_, m_, 3.); hi = FL.highpass(f_, m_, 3.)
    yes("land is NaN in both", np.isnan(lo[25, 35]) and np.isnan(hi[25, 35]))
    ok("the parts rebuild the field", (lo + hi - f_)[m_], 1e-12)
    ok("constant kept right up to the coast", FL.lowpass(5.*np.ones((ny_, nx_)), m_, 3.)[m_] - 5, 1e-11)
    ok("constant has no high-pass", FL.highpass(5.*np.ones((ny_, nx_)), m_, 3.)[m_], 1e-11)
    from functools import partial
    old = filters.masked_smooth(f_, m_*1., 3., partial(orca.smooth_2D, mask=None))
    ok("same as masked_smooth on wet points", (lo - old)[m_], 1e-12)
    wave = np.sin(2*np.pi*jj/20.)*np.cos(2*np.pi*ii/20.); oo = np.ones_like(m_)
    ok("Gaussian transfer function", (FL.lowpass(wave, oo, 2.) - np.exp(-2*(2*np.pi/20.)**2*2.**2/2)*wave)[12:-12, 12:-12], 5e-3)
    f3 = np.stack([f_, 2*f_]); ok("levels filtered alone, mask broadcast", (FL.lowpass(f3, m_, 3.)[1] - 2*lo)[m_], 1e-12)
    n_ = f_.copy(); n_[5, 5] = np.nan; yes("NaN in the field counts as invalid", np.isnan(FL.lowpass(n_, m_, 3.)[5, 5]) and np.isfinite(FL.lowpass(n_, m_, 3.)[6, 5]))
    isl = np.zeros((ny_, nx_), bool); isl[30, 40] = True; yes("under 1% of the footprint gives NaN", np.isnan(FL.lowpass(f_, isl, 12.)[30, 40]))
    hw = 12; pad = orca.halo(f_, np.array([hw, hw, hw])); wp = orca.halo(m_*1., np.array([hw, hw, hw]))
    pad = np.vstack([np.zeros((hw, pad.shape[1])), pad]); wp = np.vstack([np.zeros((hw, wp.shape[1])), wp])
    out = FL.lowpass(pad, wp > 0, 3., smooth=FL.gaussian); yes("padded tile through `smooth` has the same shape", out.shape == pad.shape)
    ok("tile and whole grid agree in the interior", (out[hw:-hw, hw:-hw] - lo)[hw:-hw, hw:-hw][m_[hw:-hw, hw:-hw]], 1e-9)
    v_ = rs.normal(size=(ny_, nx_)); yes("vector flips at the fold, scalar does not", not np.allclose(FL.lowpass(v_, oo, 3., point="V", vector=True)[-1], FL.lowpass(v_, oo, 3., point="V")[-1]))

with test("orcakit.filters.deseason", "orcakit.filters.time_lowpass"):
    t_ = np.arange(0., 1095.); om = 2*np.pi/365.25; noise_ = np.random.default_rng(1).normal(size=(t_.size, 4, 5))*0.01
    sea = 3 + 2*(t_/t_[-1]) + np.cos(om*t_) + 0.5*np.sin(2*om*t_)
    fld = sea[:, None, None] + noise_; fld[:, 0, 0] = np.nan
    r_, co = FL.deseason(fld, t_)
    yes("residual is about the noise", np.nanstd(r_) < 0.011 and abs(np.nanstd(r_) - 0.01) < 1e-3)
    ok("annual amplitude recovered", co[2, 1, 1] - 1, 5e-3); ok("semiannual amplitude recovered", co[5, 1, 1] - 0.5, 5e-3)
    yes("a NaN stays NaN", np.isnan(r_[:, 0, 0]).all())
    try:
        FL.deseason(fld[:4], t_[:4]); yes("too few snapshots refused", False)
    except ValueError:
        yes("too few snapshots refused", True)
    slow, fast = np.sin(2*np.pi*t_/200.), np.sin(2*np.pi*t_/5.)
    lp = FL.time_lowpass((slow + fast)[:, None]*np.ones((1, 3)), 1., 30.)
    ok("slow signal kept, fast removed, away from the ends", (lp[:, 1] - slow)[100:-100], 2e-3)
    b_ = np.ones((t_.size, 2)); b_[3, 0] = np.nan; yes("Butterworth leaves NaN columns as NaN", np.isnan(FL.time_lowpass(b_, 1., 30.)[:, 0]).all())

with test("orcakit.orca.coarse_grain"):
    idx_ = np.array([[0, 0, 1], [1, 3, 3]]); f_ = np.array([[1., 3., 5.], [7., 2., np.nan]]); w_ = np.array([[1., 3., 2.], [2., 5., 9.]])
    cg = orca.coarse_grain(f_, w_, idx_, (2, 2))
    ok("weighted average by hand", cg.ravel()[[0, 1, 3]] - np.array([2.5, 6., 2.]), 1e-13); yes("no weight gives NaN", np.isnan(cg[1, 0])); yes("target shape", cg.shape == (2, 2))
    c3 = orca.coarse_grain(np.stack([f_, 10*f_]), w_, idx_, (2, 2)); ok("levels averaged alone", c3[1].ravel()[[0, 1, 3]] - 10*cg.ravel()[[0, 1, 3]], 1e-12)
    z_ = orca.coarse_grain(f_, np.where(idx_ == 0, 0., w_), idx_, (2, 2)); yes("zero weight leaves points out", np.isnan(z_[0, 0]))
    wok = np.isfinite(f_); ok("area integral conserved", np.nansum(f_*w_) - np.nansum(cg*np.array([[4., 4.], [0., 5.]])), 1e-12)

with test("noisekit.running.running_stat", "noisekit.running.running_cov"):
    from noisekit import running as RN
    sn = [np.array([float(i)]) for i in range(6)]
    got = list(RN.running_stat(sn, 3, stride=2, stat="mean")); yes("stride selects windows", [i for i, _ in got] == [2, 4]); ok("window mean", got[1][1] - 3., 1e-14)
    ok("variance", next(RN.running_stat(sn, 3))[1] - 2/3, 1e-14); ok("std", next(RN.running_stat(sn, 3, stat="std"))[1] - np.sqrt(2/3), 1e-14)
    yes("default stride keeps old behaviour", [i for i, *_ in RN.rolling_stats(sn, 3)] == [2, 3, 4, 5])
    xs_ = [np.array([float(i), 2.]) for i in range(6)]; ys_ = [np.array([2.*i, np.nan]) for i in range(6)]
    cv = list(RN.running_cov(zip(xs_, ys_), 3, stride=2)); yes("covariance windows follow the stride", [i for i, _ in cv] == [2, 4])
    ok("covariance by hand: 2 var(x)", cv[0][1][0] - 2*(2/3), 1e-14); yes("NaN in y gives NaN", np.isnan(cv[0][1][1]))
    ok("covariance of x with itself is the variance", next(RN.running_cov(zip(sn, sn), 3))[1] - next(RN.running_stat(sn, 3))[1], 1e-14)


In [ ]:
with test("orcakit.filters.lowpass_tile", "orcakit.mpi_orca.gather"):
    ny_, nx_ = 60, 80; dec_ = mo.Decomp.serial_decomp(ny_, nx_); f_ = np.random.default_rng(3).normal(size=(ny_, nx_)).astype(np.float32); m_ = np.ones((ny_, nx_), bool); m_[20:30, 30:45] = False
    t_lp = filters.lowpass_tile(dec_, f_, m_, 3., point="U", vector=True); yes("interior shape kept", t_lp.shape == f_.shape); yes("land is NaN", np.isnan(t_lp[25, 35]))
    ok("same as the whole-grid lowpass away from the south edge", (t_lp - filters.lowpass(f_, m_, 3., point="U", vector=True))[15:][m_[15:]], 2e-5)
    ok("constant kept to the coast", filters.lowpass_tile(dec_, 5*np.ones((ny_, nx_), np.float32), m_, 3.)[m_] - 5, 2e-5)
    yes("serial gather returns the field", mo.gather(dec_, f_, root=0) is f_)


### `welford.*`

For the samples $x_k=k$, $k=1\dots n$: $\;\bar x=\frac{n+1}{2}$ and $\mathrm{Var}=\frac{n^2-1}{12}$. For $y=2x$, $\mathrm{Cov}(x,y)=2\mathrm{Var}(x)$. The sample variance has $n-1$ in place of $n$. A single sample makes the finalisers return NaN, as documented.

In [ ]:
from noisekit import welford as W
with test("noisekit.welford.Welford_mean_aggregate_init", "noisekit.welford.Welford_mean", "noisekit.welford.Welford_mean_finalize",
          "noisekit.welford.Welford_aggregate_init", "noisekit.welford.Welford_variance", "noisekit.welford.Welford_finalize",
          "noisekit.welford.Welford_aggregate_xy_init", "noisekit.welford.Welford_covariance", "noisekit.welford.Welford_cov_finalize"):
    n = 1000; O = np.ones((2, 3))
    a = W.Welford_mean_aggregate_init(O); b = W.Welford_aggregate_init(O); c = W.Welford_aggregate_xy_init(O)
    yes("one sample gives NaN", np.isnan(W.Welford_mean_finalize(W.Welford_mean(a, O))) and np.isnan(W.Welford_finalize(W.Welford_variance(b, O))))
    for k in range(1, n+1):
        a = W.Welford_mean(a, k*O); b = W.Welford_variance(b, k*O); c = W.Welford_covariance(c, k*O, 2*k*O)
    v = (n*n-1)/12
    ok("mean", W.Welford_mean_finalize(a) - (n+1)/2, 1e-10)
    m, pv, sv = W.Welford_finalize(b); ok("mean", m - (n+1)/2, 1e-10); ok("population variance", rerr(pv, v*O), 1e-12); ok("sample variance", rerr(sv, v*n/(n-1)*O), 1e-12)
    mx, my, vx, vy, cv = W.Welford_cov_finalize(c); ok("mean y", my - (n+1), 1e-9); ok("var x", rerr(vx, v*O), 1e-12); ok("var y", rerr(vy, 4*v*O), 1e-12); ok("cov", rerr(cv, 2*v*O), 1e-12)

### `running.rolling_stats`

With snapshots $x_k=k$ and window $w$: the mean is $k-\frac{w-1}{2}$ and the standard deviation is $\sqrt{(w^2-1)/12}$, independent of $k$. There are $n-w+1$ windows.

In [ ]:
from noisekit.running import rolling_stats
with test("noisekit.running.rolling_stats"):
    out = list(rolling_stats((k*np.ones(2) for k in range(10)), 4)); yes("window count", len(out) == 7)
    ok("index", [o[0] for o in out][0] - 3, 0); ok("mean", out[2][1] - (5-1.5), 1e-14); ok("std", out[2][2] - np.sqrt(15/12), 1e-14)

### `correlation.*`

- `efolding_lag`: for $\rho(k)=e^{-k/L}$ the 1/e crossing is $L$. Linear interpolation between integer lags over-estimates it by about $1/(8L)$, the curvature error of the exponential, so that plus 10% is the tolerance. A curve that never crosses is censored at $k_{\max}$.
- `spatial_correlation`: for $f=\cos(kx+\theta_t)$ with $\theta_t$ evenly spaced on the circle, the ensemble mean gives exactly $\rho_x(\ell)=\cos k\ell$ and $\rho_y(\ell)=1$.
- `temporal_correlation`: for $a_t=\cos\omega t$, $\rho(\ell)\to\cos\omega\ell$.

In [ ]:
from noisekit import correlation as cr
with test("noisekit.correlation.efolding_lag"):
    L0 = 5.5; rho = np.exp(-np.arange(1, 21)/L0)[:, None]; L, c = cr.efolding_lag(rho, 20); ok("L", L[0] - L0, 1.1/(8*L0)); yes("not censored", not c[0])
    L, c = cr.efolding_lag(0.9*np.ones((5, 1)), 5); ok("censored at kmax", L[0] - 5, 0); yes("flag", c[0])
    L, c = cr.efolding_lag(np.array([[np.nan], [.2]]), 2); ok("NaN skipped, crossing between lags 1 and 2", L[0] - (1 + (0.0)), 1.0)
with test("noisekit.correlation.spatial_correlation"):
    nt, ny, nx = 32, 12, 96; k = 2*np.pi*4/nx; th = 2*np.pi*np.arange(nt)/nt; x = np.arange(nx)
    f = np.cos(k*x[None]+th[:, None])[:, None]*np.ones((nt, ny, nx)); rx, ry = cr.spatial_correlation([f, f], 5)
    ok("rho_x = cos(k l)", (rx[:, 2, 10] - np.cos(k*np.arange(1, 6))), 1e-12); ok("rho_y = 1", ry[:, 2, 10] - 1, 1e-12)
    yes("NaN off the end", np.isnan(rx[4, 2, -1]) and np.isnan(ry[4, -1, 3]))
with test("noisekit.correlation.temporal_correlation"):
    nt = 20000; om = 2*np.pi/50; a = (3 + np.cos(om*np.arange(nt)))[:, None]; r = cr.temporal_correlation(a, 10)
    ok("rho = cos(omega l)", r[:, 0] - np.cos(om*np.arange(1, 11)), 2e-3)

### `structure.tensor_field`

$\mathbf D=\mathbf R_\theta\,\mathrm{diag}\big((w_{maj}/w_{min})^2,1\big)\mathbf R_\theta^T$, so $\mathrm{tr}\,\mathbf D=r^2+1$ and $\det\mathbf D=r^2$ for any $\theta$, and $\kappa^2=8\nu/w_{min}^2$.

In [ ]:
from noisekit import structure as st
with test("noisekit.structure.tensor_field"):
    r = 3.; Dxx, Dyy, Dxy, k2 = st.tensor_field(3e5, 1e5, 0.7, (2, 2))
    ok("trace", Dxx + Dyy - (r*r+1), 1e-12); ok("det", Dxx*Dyy - Dxy**2 - r*r, 1e-12); ok("kappa2", k2*1e10 - 8, 1e-12)
    ok("theta=0", np.array(st.tensor_field(3e5, 1e5, 0., (1, 1))[:3]).ravel() - [9, 1, 0], 1e-12)

### `structure.assemble_A`: the discrete operator

$A=\kappa^2M-L$ with $L=\nabla\cdot\mathbf D\nabla$, so for a plane wave $f=e^{i(k_xx+k_yy)}$ in the interior,

$$(Af)/(e_1e_2 f)\to\kappa^2+D_{xx}k_x^2+D_{yy}k_y^2+2D_{xy}k_xk_y .$$

Second-order convergence is checked. Also: $A=A^T$, $A$ is positive definite, $A\mathbf 1=\kappa^2e_1e_2$ (no-flux boundaries), and land points are decoupled from the sea.

In [ ]:
with test("noisekit.structure.assemble_A"):
    Ls = 2e6; kx, ky = 2*np.pi*3/Ls, 2*np.pi*2/Ls; E = []
    for n in (101, 201):
        d = Ls/(n-1); O = np.ones((n, n)); Dxx, Dyy, Dxy, k2 = st.tensor_field(2e5, 1e5, 0.6, (n, n))
        A, M = st.assemble_A(Dxx, Dyy, Dxy, k2, d*O, d*O, O)
        jj, ii = np.mgrid[:n, :n]*d; f = np.cos(kx*ii + ky*jj); Af = (A @ f.ravel()).reshape(n, n)/(d*d)
        ex = (k2[0, 0] + Dxx[0, 0]*kx*kx + Dyy[0, 0]*ky*ky + 2*Dxy[0, 0]*kx*ky)*f
        E.append(np.abs(Af-ex)[5:-5, 5:-5].max()/np.abs(ex).max())
    ok("symbol, relative error, n=201", E[1], 2e-3); ok("order minus 2", np.log2(E[0]/E[1]) - 2, 0.2)
    ok("symmetric", (A - A.T).max(), 1e-20)
    lam = np.linalg.eigvalsh(st.assemble_A(*st.tensor_field(2e5, 1e5, 0.6, (20, 20)), 1e4*np.ones((20, 20)), 1e4*np.ones((20, 20)), np.ones((20, 20)))[0].toarray())
    yes("positive definite", lam.min() > 0)
    n = 30; Dxx, Dyy, Dxy, k2 = st.tensor_field(1e5, 1e5, 0., (n, n)); wet = np.ones((n, n)); wet[10:15, 10:15] = 0
    A, M = st.assemble_A(Dxx, Dyy, Dxy, k2, 1e4*np.ones((n, n)), 2e4*np.ones((n, n)), wet)
    ok("A 1 = kappa2 e1 e2 with no-flux land", (A @ np.ones(n*n)) / (k2[0, 0]*2e8) - 1, 1e-10)
    ok("mass matrix", M.diagonal() - 2e8, 0)

### `structure.pcg_jacobi`, `cheby_jacobi`

Both must solve $Ax=b$ for a manufactured $x^\star$, reaching the tolerance. For the Chebyshev solver the spectral bounds of $D^{-1}A$ are computed exactly with `eigvals`.

In [ ]:
with test("noisekit.structure.pcg_jacobi", "noisekit.structure.cheby_jacobi"):
    n = 25; O = np.ones((n, n)); A, _ = st.assemble_A(*st.tensor_field(2e5, 1e5, .4, (n, n)), 2e4*O, 2e4*O, O)
    xs = np.random.default_rng(1).standard_normal(n*n); b = A @ xs; P = 1/A.diagonal()
    x, h = st.pcg_jacobi(A, b, P, tol=1e-10); ok("PCG", rerr(x, xs), 1e-7); yes("PCG reached tolerance", h[-1] < 1e-10)
    ev = np.linalg.eigvals(np.diag(P) @ A.toarray()).real
    x, h = st.cheby_jacobi(A, b, P, ev.min(), ev.max(), tol=1e-10); ok("Chebyshev", rerr(x, xs), 1e-7); yes("Chebyshev reached tolerance", h[-1] < 1e-10)

### `structure.build_filter`, `draw`, `unit_variance_scale`: Matérn field

The solution of $(\kappa^2-\Delta)u=\dot W$ in 2D is a Matérn field with $\nu=1$, whose exact variance and correlation are

$$\sigma^2=\frac{1}{4\pi\kappa^2},\qquad \rho(r)=\kappa r\,K_1(\kappa r).$$

Here $w_{maj}=w_{min}$. The exact covariance $C=A^{-1}MA^{-1}$ is computed from one solve, with no sampling, and compared at the domain centre. The Monte Carlo variance from `draw` and the scale from `unit_variance_scale` must agree within sampling error, about $\sqrt{2/N}$. A rotated elliptical filter must give a correlation whose long axis points along $\theta$.

In [ ]:
with test("noisekit.structure.build_filter", "noisekit.structure.draw", "noisekit.structure.unit_variance_scale"):
    n = 181; d = 2e4; O = np.ones((n, n)); w = 2e5; lu = st.build_filter(w, w, 0., d*O, d*O, O); kap = np.sqrt(8)/w
    c = n//2; e = np.zeros(n*n); e[c*n+c] = 1; C = lu.solve((d*d)*lu.solve(e)).reshape(n, n)
    ok("variance against 1/(4 pi kappa^2), relative", C[c, c]*4*np.pi*kap*kap - 1, 0.08)
    for ell in (5, 10, 20):
        rr = kap*ell*d; ok(f"correlation at {ell} cells against kr K1(kr)", C[c, c+ell]/C[c, c] - rr*k1(rr), 0.02)
    rng = np.random.default_rng(2); S = st.unit_variance_scale(lu, d*O, d*O, O, rng, samples=128)
    ok("MC std against exact", np.median(S[60:-60, 60:-60])/np.sqrt(C[c, c]) - 1, 0.1)
    sm = st.draw(lu, d*O, d*O, O, rng); yes("draw has the grid's shape", sm.shape == (n, n))
    dd = 1e4; n2 = 181; O2 = np.ones((n2, n2)); lu2 = st.build_filter(3e5, 1e5, np.pi/4, dd*O2, dd*O2, O2); c = n2//2
    e = np.zeros(n2*n2); e[c*n2+c] = 1; C2 = lu2.solve(dd*dd*lu2.solve(e)).reshape(n2, n2)
    yes("long axis along theta = 45 degrees", C2[c+15, c+15] > 2*C2[c-15, c+15])

## 6. Reduced-order modelling (`modredkit`)

### `pod.*`

With orthonormal $\phi_1,\phi_2$ and $X=a_1\phi_1\sin\omega t+a_2\phi_2\cos\omega t$ over an integer number of periods ($m$ snapshots), the singular values are exactly

$$\sigma_i=a_i\sqrt{m/2}.$$

Modes match up to sign, the reconstruction is exact, energy fractions are known, and weighted modes must be orthonormal: $\Phi^TW\Phi=I$. `pod_multi` on $(u,v)$ must equal `pod` on the stacked matrix.

In [ ]:
from modredkit import pod as P, modes as Mo, dmd as D
with test("modredkit.pod.pod", "modredkit.pod.reconstruct", "modredkit.pod.rank_from_energy", "modredkit.pod.pod_multi"):
    n, m = 80, 200; x = np.linspace(0, 2*np.pi, n, endpoint=False); ph1, ph2 = np.sin(x)*np.sqrt(2/n), np.cos(2*x)*np.sqrt(2/n)
    t = 2*np.pi*5*np.arange(m)/m; X = 7*ph1[:, None]*np.sin(t) + 3*ph2[:, None]*np.cos(t)
    U, S, C, mu = P.pod(X, r=2); ok("sigma_1, sigma_2", S - [7*np.sqrt(m/2), 3*np.sqrt(m/2)], 1e-11); ok("mode 1 up to sign", np.abs(U[:, 0]@ph1) - 1, 1e-12); ok("mode 2 up to sign", np.abs(U[:, 1]@ph2) - 1, 1e-12)
    ok("reconstruction", P.reconstruct(U, C, mu) - X, 1e-12); ok("mean removed", (X+5).shape[0] - n, 0)
    U5, S5, C5, mu5 = P.pod(X+5, r=2); ok("mean recovered", mu5 - 5, 1e-12)
    ok("rank, energy", P.rank_from_energy(np.array([2., 1.]), .79) - 1, 0); ok("rank, energy above first", P.rank_from_energy(np.array([2., 1.]), .81) - 2, 0)
    w = 1 + (np.arange(n) % 3); Uw, Sw, Cw, _ = P.pod(X, r=2, weights=w); ok("weighted orthonormality", Uw.T @ (w[:, None]*Uw) - np.eye(2), 1e-12)
    ok("weighted reconstruction", P.reconstruct(Uw, Cw) - (X - X.mean(1, keepdims=True)), 1e-11)
    Y = np.vstack([X, 2*X[::-1]+np.cos(3*x)[:, None]*np.sin(2*t)]); Um, Sm, Cm, _ = P.pod_multi([Y[:n], Y[n:]], r=3); U1, S1, C1, _ = P.pod(Y, r=3)
    ok("pod_multi singular values", Sm - S1, 1e-9); ok("pod_multi coeffs up to sign", np.abs(np.abs(Cm*C1).sum(1)/(S1**2)) - 1, 1e-9)

### `modes.*`

`unpack(pack(f))=f` with NaN on land. Each mask is checked by hand on $\lambda=e^{i\omega}$ values. For $\lambda=\rho e^{i\theta}$, `time_dynamics` must give $\lambda^t$, and `undamped` $e^{i\theta t}$. `reconstruct` must rebuild $x_t=\mathrm{Re}\sum b_k\phi_k\lambda_k^t$.

In [ ]:
with test("modredkit.modes.pack", "modredkit.modes.unpack", "modredkit.modes.gram", "modredkit.modes.mask_unit_circle", "modredkit.modes.mask_not_growing", "modredkit.modes.mask_period", "modredkit.modes.mask_one_per_pair", "modredkit.modes.mask_top_energy", "modredkit.modes.apply_mask", "modredkit.modes.time_dynamics", "modredkit.modes.reconstruct"):
    f = np.arange(24.).reshape(4, 2, 3); wet = np.array([[1, 1, 0], [0, 1, 1]], bool); g = 2*f; X = Mo.pack([f, g], [wet, wet]); yes("shape", X.shape == (8, 4))
    a, b = Mo.unpack(X, [wet, wet]); ok("field", a[:, wet] - f[:, wet], 0); yes("land NaN", np.isnan(a[:, ~wet]).all()); ok("second field", b[:, wet] - g[:, wet], 0)
    S2 = np.array([[1., 2.], [3., 4.]]); Z = np.array([[1., 0.], [0., 2.]])
    ok("linear gram", Mo.gram(Z) - [[1, 0], [0, 4]], 0); ok("poly gram", Mo.gram(Z, "poly", 2, 1.) - [[4, 1], [1, 25]], 0); ok("poly default scale", Mo.gram(Z, "poly")[1, 1] - (1+4/2.5)**2, 1e-13)
    lam = np.array([1.0005, 0.9, 1.01, np.exp(.2j)]); ok("unit circle", Mo.mask_unit_circle(lam) * 1 - [1, 0, 0, 1], 0); ok("not growing", Mo.mask_not_growing(lam) * 1 - [1, 1, 0, 1], 0)
    ct = 1j*2*np.pi/np.array([10., 50., 200., -50.]); ok("period band", Mo.mask_period(ct, 20, 100)*1 - [0, 1, 0, 1], 0); ok("one per pair", Mo.mask_one_per_pair(np.array([1j, -1j, 0j]))*1 - [1, 0, 1], 0)
    mds = np.array([[1., 3.], [0., 4.]]); ok("top energy: |1|*1 < |1|*5", Mo.mask_top_energy(mds, np.ones(2), 1)*1 - [0, 1], 0)
    m2, a2 = Mo.apply_mask(np.array([False, True]), mds, np.array([9, 8])); ok("apply", m2[:, 0] - [3, 4], 0); ok("apply rest", a2 - [8], 0)
    lm = np.array([.9*np.exp(.3j)]); ok("exact", Mo.time_dynamics(lm, 4)[0] - lm**np.arange(4), 1e-14); ok("undamped", Mo.time_dynamics(lm, 4, "undamped")[0] - np.exp(.3j*np.arange(4)), 1e-14)
    ph = np.array([[1+1j], [2.]]); ok("reconstruct", Mo.reconstruct(ph, np.array([2.]), lm, 4) - (ph*2*lm**np.arange(4)).real, 1e-13)

### `dmd.RelativeInformationContent`, `SVD`

For $S=(4,3,2,1)$ the cumulative content is $0.4,0.7,0.9,1$. A float threshold $0.7$ gives rank 2 and $90.0$ gives rank 3. An integer `r` means that rank.

`SVD` must return $USV^*=X$ at full rank and the best rank-$r$ approximation, with the Eckart-Young error $\|X-X_r\|_F^2=\sum_{i>r}\sigma_i^2$.

In [ ]:
with test("modredkit.dmd.RelativeInformationContent"):
    S_ = np.array([4., 3., 2., 1.]); ok("float fraction", D.RelativeInformationContent(S_, .7) - 2, 0); ok("float percent", D.RelativeInformationContent(S_, 90.) - 3, 0)
    ok("int means that rank", D.RelativeInformationContent(S_, 2) - 2, 0); ok("None keeps all", D.RelativeInformationContent(S_) - 4, 0)
with test("modredkit.dmd.SVD"):
    Xr = np.random.default_rng(3).standard_normal((30, 12)); U, S, Vh = D.SVD(Xr, None); ok("full", U*S@Vh - Xr, 1e-12)
    U, S2, Vh = D.SVD(Xr, 4); ok("Eckart-Young", np.linalg.norm(Xr - U*S2@Vh)**2 - (S[4:]**2).sum(), 1e-10); yes("rank", S2.size == 4)
    ok("float 100.0 keeps all", D.SVD(Xr, 100.0)[1].size - 12, 0)

### `dmd.exactDMD`, `exactDMD_Light`, `DefineXandXp`, `DMD_continuous_eig`

For a linear system with known eigenvalues, $x_t=\sum_kb_k\phi_k\lambda_k^t$, exact DMD must recover $\lambda_k$ to near machine precision, and $\Phi_k$ must be parallel to $\phi_k$. Here $\lambda=\{e^{\pm i\omega_1},\;0.97e^{\pm i\omega_2},\;0.9\}$. Then $\mu=\ln\lambda/\Delta t$. The `Light` version must give the same eigenvalues.

In [ ]:
rng = np.random.default_rng(4); n, m = 60, 50; dt = 3.
w1, w2 = 0.3, 0.8; LAM = np.array([np.exp(1j*w1), np.exp(-1j*w1), .97*np.exp(1j*w2), .97*np.exp(-1j*w2), .9])
Q, _ = np.linalg.qr(rng.standard_normal((n, 5))); ph1 = Q[:, 0]+1j*Q[:, 1]; ph2 = Q[:, 2]+1j*Q[:, 3]
PHI = np.column_stack([ph1, ph1.conj(), ph2, ph2.conj(), Q[:, 4]]); BB = np.array([2+1j, 2-1j, 1-.5j, 1+.5j, 1.5])
SNAP = (PHI*BB @ (LAM[:, None]**np.arange(m)[None])).real
def match(a, b):
    "Largest distance after pairing each element of a with its closest in b."
    return max(np.min(np.abs(b - x)) for x in a)
with test("modredkit.dmd.exactDMD", "modredkit.dmd.exactDMD_Light", "modredkit.dmd.DefineXandXp", "modredkit.dmd.DMD_continuous_eig"):
    Xb, Xa = D.DefineXandXp(SNAP); ok("X and X' shifted", Xb[:, 1:] - Xa[:, :-1], 0)
    mds, ev, nm = D.exactDMD(Xb, Xa, 5); ok("eigenvalues", match(LAM, ev), 1e-10); yes("5 modes", nm == 5)
    i = np.argmin(np.abs(ev - LAM[0])); ok("mode parallel to phi_1", 1 - np.abs(np.vdot(mds[:, i], ph1))/(np.linalg.norm(mds[:, i])*np.linalg.norm(ph1)), 1e-10)
    ok("Light gives the same eigenvalues", match(LAM, D.exactDMD_Light(SNAP, 5)[1]), 1e-10)
    ct = D.DMD_continuous_eig(np.array([np.exp(.6j)*.5]), dt); ok("mu = ln(lambda)/dt", ct - (np.log(.5)+.6j)/dt, 1e-14)

### Amplitudes, dynamics and mode bookkeeping

- `DMD_amplitudes_pinv`: $b=\Phi^+x_0$ recovers the known $b$.
- `DMD_amplitudes_optimal`: minimises $\|X-\Phi\,\mathrm{diag}(b)\,\mathrm{Vand}\|_F$; on noise-free data it must recover the same $b$. Added noise must leave it closer to the truth than `pinv`, which sees one snapshot.
- `DMD_amplitudes_2014Jovanovich`: the same problem in POD coordinates, so its $b$ belongs to the projected modes $UW$. Exact modes are $\Phi=UW\Lambda$, so with those the amplitudes are $b/\lambda$.
- `DMD_dynamics_complex`: $\mathrm{Re}\,\Phi e^{\mu t}$ must rebuild the snapshots. `DMD_dynamics_noPairs` stores one mode per pair, and for $\lambda$ on the unit circle gives $2\mathrm{Re}(\phi e^{i\omega t})$.
- `getConjugatePairsMask`: keeps one of each conjugate pair, and all real modes.
- `CleanUnitCircle` and `RemGtT`: drop modes off the circle. `RemGtT` takes a threshold that must change the result.
- `SortByFrequency`: sorts by $|\mathrm{Im}\,\mu|$, as its docstring says, whatever the damping.

In [ ]:
with test("modredkit.dmd.DMD_amplitudes_pinv", "modredkit.dmd.DMD_amplitudes_optimal", "modredkit.dmd.DMD_amplitudes_2014Jovanovich"):
    amp, ap = D.DMD_amplitudes_pinv(PHI, SNAP[:, 0], apply=True); ok("pinv recovers b", amp - BB, 1e-10); ok("apply", ap - PHI*BB, 1e-10)
    ok("optimal recovers b", D.DMD_amplitudes_optimal(PHI, LAM, SNAP) - BB, 1e-8)
    b2, en = D.DMD_amplitudes_optimal(PHI, LAM, SNAP, return_energy=True); yes("energy is real and positive", (en > 0).all())
    nois = np.random.default_rng(5).standard_normal(SNAP.shape)*.05
    yes("optimal beats pinv on noisy data", np.linalg.norm(D.DMD_amplitudes_optimal(PHI, LAM, SNAP+nois)-BB) < np.linalg.norm(D.DMD_amplitudes_pinv(PHI, (SNAP+nois)[:, 0])-BB))
    mds, ev, nm, U, S_, Vh, Wv = D.exactDMD(SNAP[:, :-1], SNAP[:, 1:], 5, SVD_out=True)
    bj = D.DMD_amplitudes_2014Jovanovich(S_, Vh.conj().T, Wv, ev); ok("Jovanovic reconstructs with projected modes U W", (((U@Wv)*bj)@(ev[:, None]**np.arange(m-1)[None])).real - SNAP[:, :-1], 1e-7)
    ok("exact modes Phi need b/lambda: Phi = U W Lambda", ((mds*(bj/ev))@(ev[:, None]**np.arange(m-1)[None])).real - SNAP[:, :-1], 1e-7)
with test("modredkit.dmd.DMD_dynamics_complex", "modredkit.dmd.DMD_dynamics_noPairs"):
    rec, ct = D.DMD_dynamics_complex(PHI*BB, LAM, dt, m); ok("complex dynamics", rec - SNAP, 1e-10)
    lm = np.array([np.exp(1j*w1)]); ph = (ph1*(1+.5j))[:, None]; t = np.arange(20)
    rec2, _ = D.DMD_dynamics_noPairs(ph, lm, 1., 20); ok("no-pairs dynamics", rec2 - 2*(ph*lm[:, None]**t[None]).real, 1e-11)
    lm3 = np.array([.97*np.exp(.3j), .9+0j]); ph3 = np.array([[1+.5j, 2.], [.3-.2j, -1.]]); rec3, _ = D.DMD_dynamics_noPairs(ph3, lm3, 1., 20)
    ok("damped pair plus real mode", rec3 - (2*(ph3[:, :1]*lm3[:1, None]**t[None]).real + (ph3[:, 1:]*lm3[1:, None]**t[None]).real), 1e-11)
with test("modredkit.dmd.getConjugatePairsMask"):
    mk, nn = D.getConjugatePairsMask(np.array([1+2j, 1-2j, .1+0j, .2+0j])); ok("pair kept once, both real modes kept", nn - 3, 0); yes("the kept pair member has Im > 0", mk[0] and not mk[1])
with test("modredkit.dmd.RemGtT", "modredkit.dmd.CleanUnitCircle", "modredkit.dmd.SortByFrequency", "modredkit.dmd.ShiftStack"):
    lam = np.array([1.0, .95, 1.2]); ct = np.log(lam.astype(complex)); ph = np.eye(3); am = np.array([1., 2., 3.])
    out = D.RemGtT(ph, lam, ct, am); ok("RemGtT default drops only the growing mode", out[3] - [1., 2.], 0)
    ok("RemGtT threshold honoured: 0.96 keeps only 0.95", D.RemGtT(ph, lam, ct, am, treshold=.96)[3] - [2.], 0)
    out = D.CleanUnitCircle(ph, ct, lam, am, .1); ok("CleanUnitCircle", out[3] - [1., 2.], 0); ok("count", out[4] - 2, 0)
    ct2 = np.array([-10+.1j, 0+1j]); s = D.SortByFrequency(np.eye(2), ct2, ct2, ct2); ok("sorted by |Im mu|", s[1].imag - [.1, 1], 0)
    Sx = np.arange(1., 7.)[None]; ss = D.ShiftStack(Sx, 6, 2); ok("Hankel rows", ss - [[1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 5, 6]], 0)
    yy = np.cos(.4*np.arange(40.))[None]; hh = D.ShiftStack(yy, 40, 5); ok("delay embedding finds omega for a scalar", match(np.exp([.4j, -.4j]), D.exactDMD(hh[:, :-1], hh[:, 1:], 2)[1]), 1e-10)

### `st_unravel`, `s_unravel`, `st_ravel`, `s_ravel`

These are inverses: $\mathrm{ravel}(\mathrm{unravel}(f))=f$, and column $t$ of the matrix holds snapshot $t$.

In [ ]:
with test("modredkit.dmd.st_unravel", "modredkit.dmd.s_unravel", "modredkit.dmd.st_ravel", "modredkit.dmd.s_ravel"):
    f = np.arange(120.).reshape(2, 3, 4, 5); M, nt, ns = D.st_unravel(f, shape=True); ok("column t = snapshot t", M[:, 1] - f[1].ravel(), 0); ok("inverse", D.st_ravel(M, nt, ns) - f, 0)
    v, ns = D.s_unravel(f[0], shape=True); ok("space only", D.s_ravel(v, ns) - f[0], 0)

### `kernelDMD`: Koopman eigenvalues

With a **linear** kernel it must match exact DMD. With a **degree-2 polynomial** kernel on $x_{t+1}=\rho R_\theta x_t$, the feature space spanned by $\{1,x_1,x_2,x_1^2,x_1x_2,x_2^2\}$ is invariant, so the Koopman eigenvalues are known in closed form:

$$\{1,\;\rho e^{\pm i\theta},\;\rho^2e^{\pm2i\theta},\;\rho^2\}.$$

The modes `Xi` must rebuild the linear observable: $x_t=\Re\,\Xi\,\psi_t$, tested through $X\approx\Xi\,\mathrm{diag}(\ldots)$ in the rank-2 linear case.

In [ ]:
with test("modredkit.dmd.kernelDMD"):
    G = SNAP.T @ SNAP; out = D.kernelDMD(G, SNAP[:, :-1].T, dt, 5); ok("linear kernel eigenvalues", match(LAM, out[2]), 1e-7); ok("continuous", match(np.log(LAM)/dt, out[1]), 1e-7)
    rho, th = .98, .35; R = rho*np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]]); xs = [np.array([1., .5])]
    for _ in range(60): xs.append(R@xs[-1])
    Z = np.array(xs).T; Gp = Mo.gram(Z, "poly", 2, 1.); ev = D.kernelDMD(Gp, Z[:, :-1].T, 1., 6, noModes=True)[2]
    ex = np.array([1, rho*np.exp(1j*th), rho*np.exp(-1j*th), rho**2*np.exp(2j*th), rho**2*np.exp(-2j*th), rho**2])
    ok("polynomial kernel: closed-form Koopman eigenvalues", match(ex, ev), 1e-6)
    Gl = Z.T@Z; ml, ctl, evl, *_ = D.kernelDMD(Gl, Z[:, :-1].T, 1., 2)
    bb = np.linalg.lstsq(ml, Z[:, 0], rcond=None)[0]; ok("modes span the data: rebuild", ((ml*bb)@(evl[:, None]**np.arange(40)[None])).real - Z[:, :40], 1e-6)

### `mrDMD`

Level $\ell$ cuts the record into $2^\ell$ windows, so with $L=3$ there are $1+2+4=7$ windows. A pure oscillation must give the same $\lambda$ in every window, amplitudes fitted to each window's first snapshot, and damped modes kept. A growing mode must be dropped.

In [ ]:
with test("modredkit.dmd.mrDMD"):
    n, m = 40, 160; Q, _ = np.linalg.qr(np.random.default_rng(6).standard_normal((n, 3))); t = np.arange(m)
    Xo = Q[:, 0:1]*np.cos(.5*t) + Q[:, 1:2]*np.sin(.5*t)
    mds, ldt, lct, amp, idx = D.mrDMD(Xo, 2, 1., 3); yes("seven windows", len({(a, b) for a, b, _ in idx}) == 7); yes("2 modes per window", idx.shape[0] == 14)
    ok("lambda in all windows", np.abs(np.abs(ldt.imag) - np.sin(.5)).max(), 1e-10); ok("omega", np.abs(lct.imag) - .5, 1e-10)
    Xd = Q[:, 0:1]*.9**t; yes("damped mode kept", D.mrDMD(Xd, 1, 1., 1)[1].size == 1)
    Xg = Q[:, 0:1]*1.05**t; yes("growing mode dropped", D.mrDMD(Xg, 1, 1., 1)[1].size == 0)

### `PlotAmplitudes`, `PlotCircle`, `SaveNetCDF`

These are inherited classes and plots. The plots have no analytic result, so they are only run. For the classes: the constructor must recover $\lambda$ like `exactDMD`, sort and clean must keep the bookkeeping consistent, and `ToSpatial` must rebuild $\Phi b$ on the physical grid. `DMD_phyModes` must scale its fields and split by frequency without losing any mode.

In [ ]:
with test("modredkit.dmd.PlotAmplitudes", "modredkit.dmd.PlotCircle"):
    ct = np.log(LAM)/dt; D.PlotAmplitudes(ct, BB); D.PlotCircle(ct, LAM); plt.close("all"); yes("plots ran", True)
with test("modredkit.dmd.SaveNetCDF"):
    def res(sc):
        O = (1+2j)*sc*np.ones((2, 1, 3, 4)); return types.SimpleNamespace(nm=2, ny=3, nx=4, nz=1, ct_eig=np.array([.1j, .2j]), Umod=O, Vmod=2*O, Wmod=3*O)
    p = os.path.join(TMP, "sv.nc"); D.SaveNetCDF(res(1.), res(2.), np.ones((1, 3, 4)), 2*np.ones((1, 3, 4)), p)
    with netCDF4.Dataset(p) as nc:
        ok("corrected mode", nc["vmode_imag_c"][:] - 4, 0); ok("uncorrected mode", nc["umode_real_r"][:] - 2, 0); ok("omega", nc["omega_r"][:] - [.1, .2], 0)

## 7. Data readers (`datakit`)

### `_read.*` on a small synthetic archive

File dates are parsed from names, date filters are inclusive, values above the limit and non-finite values are cleaned, levels and boxes are honoured, and land reads as NaN. `stream_files` must refuse mismatched dates.

In [ ]:
from datakit import _read as R, cmems, glorys12
with test("datakit._read.file_date", "datakit._read.find_files", "datakit._read.clean", "datakit._read.read_one", "datakit._read.stream_files", "datakit._read.read_stack", "datakit._read.read_mesh"):
    ok("daily date", ["x_2015-03-07.nc" in "x_2015-03-07.nc", R.file_date("a/x_2015-03-07.nc") == "2015-03-07"], 1); yes("daily", R.file_date("a/x_2015-03-07.nc") == "2015-03-07"); yes("monthly", R.file_date("x_2015-03.nc") == "2015-03"); yes("compact", R.file_date("x_201503.nc") == "2015-03")
    root = os.path.join(TMP, "arch")
    for v, sc in (("uu", 1.), ("vv", 2.)):
        for d in (1, 2, 3):
            a = np.ma.array(sc*d*np.ones((1, 3, 4, 5)), mask=False); a[0, :, 0, 0] = np.ma.masked
            p = os.path.join(root, v); os.makedirs(p, exist_ok=True)
            with netCDF4.Dataset(os.path.join(p, f"{v}_2015-01-0{d}.nc"), "w") as nc:
                for dn, s in (("time_counter", 1), ("depthu", 3), ("y", 4), ("x", 5)): nc.createDimension(dn, s)
                nc.createVariable(v, "f4", ("time_counter", "depthu", "y", "x"), fill_value=1e20)[...] = a
    fl = R.find_files(root, "uu", "2015-01-02", "2015-01-03"); yes("dates inclusive", len(fl) == 2)
    yes("month filter", len(R.find_files(root, "uu", "2015-01", "2015-01")) == 3)
    ok("clean", np.nan_to_num(R.clean(np.array([1., 11., np.nan, -3.]), 10., -9)) - [1, -9, -9, -3], 0)
    s = R.read_stack(fl, "uu", level=1, box=(1, 3, 2, 5)); yes("level and box", s.shape == (2, 2, 3)); ok("value", s[1] - 3, 0)
    s = R.read_stack(fl, "uu", level=None); yes("all levels", s.shape == (2, 3, 4, 5)); yes("land NaN", np.isnan(s[0, 0, 0, 0]))
    it = list(R.stream_files([fl, R.find_files(root, "vv", "2015-01-02", "2015-01-03")], ("uu", "vv"))); ok("stream ratio", it[0][2][2, 2]/it[0][1][2, 2] - 2, 0); yes("stream date", it[0][0] == "2015-01-02")
    try: list(R.stream_files([fl, R.find_files(root, "vv", "2015-01-01", "2015-01-02")], ("uu", "vv"))); yes("mismatch refused", False)
    except ValueError: yes("mismatch refused", True)
    mp = ncfile("mesh.nc", dict(t=1, y=2, x=2), e1t=(("t", "y", "x"), np.ones((1, 2, 2)))); ok("mesh squeezed", R.read_mesh([mp], "e1t").shape[0] - 2, 0)

### `glorys04`, `cmems`, `glorys12`: real files, one small read

The only analytic check available without trusting earlier output is geometry. **Summing $e_1e_2$ over the unique cells must give the area of the sphere that the grid covers.** These grids stop at a latitude $\varphi_s$ south of which there is only Antarctic land, so the expected area is $2\pi a^2(1+\sin|\varphi_s|)$, checked to 0.5%. Each reader must also return the requested shape from a small box of one snapshot.

In [ ]:
AREA = 4*np.pi*6371e3**2
for name, mod, var, kw in (("glorys04", glorys04, "vozocrtx", dict(start="2015-01-01", end="2015-01-01")), ("cmems", cmems, "vozocrtx", dict(start="2015-01", end="2015-01")), ("glorys12", glorys12, "sossheig", dict(start="2015-01", end="2015-01"))):
    with test(f"datakit.{name}.files", f"datakit.{name}.read", f"datakit.{name}.stream", f"datakit.{name}.mesh"):
        fl = mod.files(var, kw["start"], kw["end"]); yes("one file", len(fl) == 1)
        s = mod.read(var, level=0, box=(300, 310, 500, 512), **kw); yes("box shape", s.shape == (1, 10, 12))
        d, f = next(mod.stream(var, level=0, box=(300, 310, 500, 512), **kw)); ok("stream equals read", np.nan_to_num(f - s[0]), 0)
        e1, e2, lat_ = mod.mesh("e1t", "e2t", "nav_lat"); south = np.deg2rad(lat_.min()); print(f"   grid starts at {np.rad2deg(south):.1f} degrees")
        ok("area of the sphere north of the grid's south edge, relative", np.sum((e1*e2)[:-2, 1:-1])/(AREA*(1+np.sin(np.abs(south)))/2) - 1, .005)

## 8. GYRE helpers (`gyrekit`)

- `area_int` with factors $\tfrac12$ is the trapezoid rule: for $p\equiv1$ on $n_y\times n_x$ points it gives $(n_y-1)(n_x-1)$.
- `coarse_grain_P` is documented as keeping the area integral, so $\int_{\text{coarse}}=\int_{\text{fine}}/r^2$.
- `jet_mask` has a known answer.
- `velocity_summary`: for $u=U_0+a\cos(2\pi f_0t)$, $v=0$, uniform in space: $\mathrm{MKE}=\tfrac12U_0^2$, $\mathrm{EKE}=\tfrac14a^2$, and the PSD peaks at $f_0$.
- `find_overlap` is marked "Original logic, unverified" and has no definition that can be tested analytically, so it is only run.

In [ ]:
from gyrekit import grids as Gg, runs as Gr, diag as Gd
with test("gyrekit.grids.area_int", "gyrekit.grids.coarse_grain_P", "gyrekit.grids.jet_mask", "gyrekit.grids.find_overlap"):
    ok("trapezoid of 1", Gg.area_int(np.ones((11, 21)), .5, .5) - 200, 1e-12)
    jj, ii = np.mgrid[:11, :21].astype(float); ok("trapezoid of x", Gg.area_int(ii, .5, .5) - 2000, 1e-11)
    rat = 3; ph = 2*np.ones((3*20+1, 3*30+1)); pl = Gg.coarse_grain_P(ph, rat); ok("area integral kept, relative", Gg.area_int(pl, .5, .5)/(Gg.area_int(ph, .5, .5)/rat**2) - 1, 1e-10)
    lat = np.arange(10., 50)[:, None]*np.ones((40, 60)); lon = np.arange(-115., -55)[None]*np.ones((40, 60)); jm = Gg.jet_mask(lat, lon, 10., 10.)
    ok("edges zero", [jm[-15:].max(), jm[:, :15].max()], 0); yes("inside the box", jm[int(29-10), int(-85+115)] == 1); yes("outside", jm[0, 30] == 0)
    hl = np.arange(38.)[:, None]*np.ones((38, 58)); hn = np.arange(58.)[None]*np.ones((38, 58)); ll = np.arange(20.)[:, None]*np.ones((20, 30)); ln = np.arange(30.)[None]*np.ones((20, 30)); c, r = Gg.find_overlap(hl, hn, ll, ln); yes("find_overlap ran (no analytic contract)", c.ndim == 1 and r.ndim == 1)
with test("gyrekit.diag.velocity_summary"):
    nt, ny, nx = 240, 32, 32; f0 = 1/24; t = np.arange(nt); U0, a = 2., 1.; u = (U0 + a*np.cos(2*np.pi*f0*t))[:, None, None]*np.ones((nt, ny, nx)); v = 0*u
    r = Gd.velocity_summary(u, v, 1., 1., 1., delay=0, nb=2); I = (slice(2, -2), slice(2, -2))
    ok("MKE", r["mke"][I] - .5*U0*U0, 1e-12); ok("EKE", r["eke"][I] - .25*a*a, 1e-12); ok("PSD peak", r["freq"][np.argmax(r["psd_eke"])] - f0, 1e-12)
    ok("PSD integrates to EKE", r["psd_eke"].sum()*(r["freq"][1]-r["freq"][0]) - .25*a*a, 1e-10); ok("no ring energy for a uniform flow", r["spec_ke"], 1e-20)
with test("gyrekit.runs.experiments", "gyrekit.runs.gridfile", "gyrekit.runs.read_field", "gyrekit.runs.read_spacing"):
    ex = Gr.experiments("R3"); yes("experiments found", len(ex) > 0); e0 = next(iter(ex.values())); yes("all file kinds", set(e0["files"]) == set(Gr.KINDS)); yes("no double slash", "//" not in e0["files"]["uvelfile"][0])
    yes("grid path", Gr.gridfile("R9").endswith("domain_cfg_R9.nc"))
    p = ncfile("g1.nc", dict(t=2, z=3, y=2, x=2), q=(("t", "z", "y", "x"), np.arange(24.).reshape(2, 3, 2, 2))); out = Gr.read_field([p, p], "q", levels=[1]); yes("concatenated and level picked", out.shape == (4, 1, 2, 2)); ok("value", out[0, 0, 0, 0] - 4, 0)
    p = ncfile("g2.nc", dict(t=1, y=2, x=2), e1t=(("t", "y", "x"), 5*np.ones((1, 2, 2))), e2t=(("t", "y", "x"), 7*np.ones((1, 2, 2)))); ok("spacing", np.array(Gr.read_spacing(p)) - [5, 7], 0)

### Common sources and the file convention (`datakit._source`, `datakit._write`)

A small fake mesh of $n_z=3$ levels stands in for a dataset. With a flat bottom, $e_{3u}=e_{3v}=e_{3t,0}$ on wet points, and $\sum_k u'_k e_{3k}=0$ with $\hat u+u'_k=u_k$. Fields written with `write_field` must read back through `derived` under the common names, with their type, time and depth kept.


In [ ]:
import importlib.util
from datakit import _source as SRC, _write as WR
import sys as _s; _s.path.insert(0, os.path.join(ROOT, "..", "preprocessing"))
spec = importlib.util.spec_from_file_location("baroclinic", os.path.join(ROOT, "..", "preprocessing", "baroclinic.py"))
BC = importlib.util.module_from_spec(spec); spec.loader.exec_module(BC)
with test(*[f"datakit._source.Source.{m}" for m in ("__init__", "files", "read", "stream", "groups", "mesh", "grid", "vertical", "coords", "depth", "time_hours", "snapshot")], "datakit._source.source", "datakit._source.derived", "datakit._write.write_field", "datakit._write.provenance", "datakit._write.history", "preprocessing.baroclinic.split", "orcakit.grid_c.vertical_velocity", "datakit._source.Source.point_mask", "datakit._source.Source.point_coords", "orcakit.paths.get"):
    from orcakit import paths as PA
    toml = os.path.join(TMP, "paths.toml")
    with open(toml, "w") as t_: t_.write('one = "~/x"\n')
    saved = os.environ.get("LU_PATHS"); os.environ["LU_PATHS"] = toml
    yes("location read, with ~ expanded", PA.get("one") == os.path.expanduser("~/x"))
    yes("unset location gives a placeholder that names it", "two" in PA.get("two") and "not set" in PA.get("two"))
    if saved is None: del os.environ["LU_PATHS"]
    else: os.environ["LU_PATHS"] = saved
    nz, ny, nx = 3, 12, 16; e3 = np.array([10., 20., 30.]); mesh = os.path.join(TMP, "stage0_mesh.nc")
    tm = np.ones((nz, ny, nx), "i1"); tm[:, 5, 6] = 0
    um = tm*np.roll(tm, -1, axis=2); vm = tm*np.roll(tm, -1, axis=1)
    with netCDF4.Dataset(mesh, "w") as nc:
        for d, s in (("t", 1), ("z", nz), ("y", ny), ("x", nx)): nc.createDimension(d, s)
        nc.createVariable("e3t_0", "f8", ("t", "z"))[:] = e3
        nc.createVariable("e3t_ps", "f8", ("t", "y", "x"))[:] = e3[-1]
        for n_, a in (("tmask", tm), ("umask", um), ("vmask", vm)): nc.createVariable(n_, "i1", ("t", "z", "y", "x"))[:] = a
        for n_ in ("gphit", "glamt", "gphiu", "glamu", "e1t", "e2t", "e1u", "e2u", "e1v", "e2v", "e1f", "e2f"): nc.createVariable(n_, "f8", ("t", "y", "x"))[:] = 1000.
    S = SRC.Source("fake", os.path.join(TMP, "stage0_raw"), [mesh], "fk", 86400.)
    e3u, e3v, u_m, v_m = S.vertical(np.float64)
    yes("masks returned as boolean", u_m.dtype == bool and v_m.dtype == bool)
    ok("e3u is e3t_0 on wet points", (e3u - e3[:, None, None])[:, 1:-2, 1:-2][u_m[:, 1:-2, 1:-2]], 1e-12)
    ok("e3v is e3t_0 on wet points", (e3v - e3[:, None, None])[:, 1:-2, 1:-2][v_m[:, 1:-2, 1:-2]], 1e-12)
    g_ = S.grid(); yes("grid names the boundary and the metrics", g_["boundary"] == "t-fold" and g_["e1u"].shape == (ny, nx) and g_["lat"].shape == (ny, nx))
    yes("mesh reads by name", S.mesh("e3t_0").shape == (nz,))
    try:
        SRC.Source("f", "", [mesh], "f", 1., boundary="closed").vertical(); yes("other boundaries refused", False)
    except NotImplementedError:
        yes("other boundaries refused", True)
    u = 0.1*np.arange(1., nz*ny*nx + 1).reshape(nz, ny, nx)/(nz*ny*nx); u[:, 5, 6] = np.nan
    bt, bc = BC.split(u, e3u, u_m)
    w = e3u*u_m; ok("baroclinic part has zero weighted depth mean", np.nansum(np.nan_to_num(bc)*w, axis=0), 1e-11)
    wet = u_m.all(axis=0); ok("the parts rebuild the field", (bc + bt[None] - u)[:, wet], 1e-12)
    ok("barotropic is the weighted mean, hand value", bt[0, 0] - np.sum(u[:, 0, 0]*e3)/e3.sum(), 1e-12)
    yes("land is NaN in both parts", np.isnan(bc[0, 5, 6]) and np.isnan(bt[5, 6]) and (np.isnan(bc) == ~u_m).all())
    attrs = WR.provenance(source="fake"); yes("provenance has commit, command, date and extras", all(k in attrs for k in ("git_commit", "command", "created", "source")))
    lat, lon = np.zeros((ny, nx)), np.zeros((ny, nx)); dep = np.array([5., 20., 45.]); root = S.root
    for i, date in enumerate(("2015-01-01", "2015-01-02")):
        for var, sign in (("u", 1.), ("v", -1.)):
            p = WR.write_field(root, var, date, sign*(i + 1)*u, lat, lon, 100. + i, "fk", dep, np.float64 if var == "v" else np.float32, attrs)
    yes("path follows the convention", p.endswith(os.path.join("vomecrty", "vomecrty_fk_2015-01-02.nc")) and not glob.glob(p + ".tmp*"))
    with netCDF4.Dataset(p) as nc:
        yes("file convention", nc["vomecrty"].dimensions == ("deptht", "y", "x") and nc["time_counter"].shape == () and np.isnan(nc["vomecrty"]._FillValue) and nc["vomecrty"].filters()["zlib"] and nc["vomecrty"].dtype == np.float64 and nc.source == "fake")
    bt_path = WR.write_field(os.path.join(TMP, "stage0_bt"), "v", "2015-01-01", bt, lat, lon, 100., "fk")
    with netCDF4.Dataset(bt_path) as nc: yes("2D field has no depth", nc["vomecrty"].dimensions == ("y", "x"))
    both = WR.write_field(os.path.join(TMP, "stage0_split"), "v", "2015-01-01", bc, lat, lon, 100., "fk", dep, np.float64, attrs, {"vobtcrty": bt})
    Sp = SRC.derived(os.path.join(TMP, "stage0_split"), S)
    yes("both parts share one file", Sp.files("vbt") == Sp.files("v") == [both])
    with netCDF4.Dataset(both) as nc: yes("barotropic part is 2D, baroclinic 3D, same type", nc["vobtcrty"].dimensions == ("y", "x") and nc["vomecrty"].dimensions == ("deptht", "y", "x") and nc["vobtcrty"].dtype == np.float64)
    ok("barotropic part reads back by common name", np.nan_to_num(Sp.read("vbt")[0] - np.float32(bt)), 1e-6)
    ok("baroclinic part reads back by common name", np.nan_to_num(Sp.read("v", level=None)[0] - np.float32(bc)), 1e-6)
    import _common as CM
    yes("resume accepts the same precision", not CM.incompatible(both, ("vomecrty", "vobtcrty"), np.float64))
    yes("resume refuses another precision", CM.incompatible(both, ("vomecrty", "vobtcrty"), np.float32))
    yes("resume refuses a file with one part", CM.incompatible(p, ("vomecrty", "vobtcrty"), np.float64))
    WV = lambda a, b, m: grid_c.vertical_velocity(a, b, exchange=orca.halo, **m)
    yes("point mask by name", S.point_mask("U").dtype == bool)
    pc = S.point_coords("U"); yes("point coordinates from the mesh", len(pc) == 2 and pc[0].shape == (ny, nx))
    gr = S.grid(); M_ = dict(e3u=e3u, e3v=e3v, umask=np.ones_like(u_m), vmask=np.ones_like(v_m), tmask=np.ones_like(u_m), e1t=gr["e1t"], e2t=gr["e2t"], e1v=gr["e1v"], e2u=gr["e2u"])
    M_["e3u"] = M_["e3v"] = e3[:, None, None]*np.ones((nz, ny, nx))
    w_ = WV(np.full((nz, ny, nx), 0.3), np.full((nz, ny, nx), 0.0), M_)
    ok("uniform flow in a periodic channel has no w", w_[:, 1:, 1:-1], 1e-12)
    jj_ = np.arange(ny)[None, :, None]*np.ones((nz, ny, nx)); w_ = WV(np.zeros((nz, ny, nx)), 1e-3*jj_, M_)
    ok("constant divergence gives w by hand, top of the column", w_[0, 3:-3, 3:-3] + 1e-6*e3[:-1].sum(), 1e-12)
    yes("w is NaN on land", np.isnan(WV(np.zeros((nz, ny, nx)), np.zeros((nz, ny, nx)), {**M_, "tmask": tm.astype(bool)})[0, 5, 6]))
    dep_ = WR.write_field(os.path.join(TMP, "stage0_w"), "w", "2015-01-01", w_, lat, lon, 100., "fk", dep, depth_name="depthw")
    with netCDF4.Dataset(dep_) as nc: yes("w files use W levels", nc["vovecrtz"].dimensions == ("depthw", "y", "x"))
    raw_ = os.path.join(TMP, "raw_hist.nc")
    with netCDF4.Dataset(raw_, "w") as nc: nc.history = "made upstream"
    h1 = WR.provenance("PP1 baroclinic: split", raw_)["history"]
    yes("first step: its line, then the raw file and its own history", h1.splitlines()[0].endswith(" | " + os.path.basename(sys.argv[0]) + " " + " ".join(sys.argv[1:])) and "PP1 baroclinic: split" in h1.splitlines()[0] and h1.splitlines()[1:] == ["source file: raw_hist.nc", "made upstream"])
    mid_ = WR.write_field(os.path.join(TMP, "hist"), "u", "2015-01-01", bt, lat, lon, 100., "fk", attrs=WR.provenance("PP1 baroclinic: split", raw_))
    h2 = WR.provenance("PP4 spatial_filter: low", mid_)["history"].splitlines()
    yes("second step goes on top of the first, newest first", "PP4" in h2[0] and "PP1" in h2[1] and h2[2] == "source file: raw_hist.nc")
    yes("no git commit in history", all("git" not in l for l in h2))
    h3 = WR.provenance("PP7 running: cov", {"u": mid_, "v": raw_})["history"].splitlines()
    yes("two inputs keep both histories, indented", h3[1] == "inputs: u" and h3[2].startswith("  ") and "inputs: v" in h3 and "  source file: raw_hist.nc" in h3)
    yes("without a step, no history", "history" not in WR.provenance(source="x"))
    yes("output path", BC.output_path("d", "vomecrty", "fk", "1993-01") == os.path.join("d", "split", "vomecrty", "vomecrty_fk_1993-01.nc"))
    D = SRC.derived(root, S); yes("derived keeps the grid, tag and step", D.mesh_files == [mesh] and D.tag == "fk" and D.dt == 86400. and D.name == "stage0_raw")
    yes("common and source names find the same files", D.files("u") == D.files("vozocrtx") and len(D.files("u", "2015-01-02")) == 1)
    r = D.read("u", level=None); ok("read back, single precision", np.nan_to_num(r[1] - np.float32(2*u)), 2e-6); yes("read shape", r.shape == (2, nz, ny, nx))
    ok("double precision survives the round trip", np.nan_to_num(D.snapshot(D.files("v")[0], "v", np.float64) + u), 1e-14)
    got = list(D.stream("u", "v", level=1)); yes("stream by common names", [g[0] for g in got] == ["2015-01-01", "2015-01-02"] and got[0][1].shape == (ny, nx))
    os.remove(D.files("v", "2015-01-02")[0]); gr, miss = D.groups("u", "v"); yes("groups keeps common dates and names the others", [g[0] for g in gr] == ["2015-01-01"] and miss == ["2015-01-02"])
    f0 = D.files("u")[0]; ok("time kept", D.time_hours(f0) - 100., 1e-9); ok("depth kept", D.depth(f0) - dep, 1e-5); yes("coordinates kept", D.coords(f0)[0].shape == (ny, nx))
    other = os.path.join(TMP, "stage0_time.nc")
    with netCDF4.Dataset(other, "w") as nc:
        nc.createDimension("time_counter", 1); t = nc.createVariable("time_counter", "f8", ("time_counter",)); t.units, t.calendar = "seconds since 1950-01-02", "gregorian"; t[:] = 3600.
    ok("other time units convert: one day and one hour", D.time_hours(other) - 25., 1e-9)
    for name_ in ("glorys04", "glorys12", "cmems"): yes(f"source finds {name_}", SRC.source(name_).name == name_)


## 9. Pipelines on fake datasets

Pipelines read through a dataset module. A small fake one stands in here, whose fields have known statistics, so the whole chain runs in seconds: reading, statistics, filtering, sampling onto a coarse grid, writing the file.

### `variance.run` (V1-V3)

With $u_k=sk$, $v_k=2sk$ on $k=1\dots n$, with $s=0.1$ so the speeds stay under the 10 m/s limit of `datakit.clean`:

$$\bar u=s\tfrac{n+1}{2},\quad \mathrm{Var}\,u=s^2\tfrac{n^2-1}{12},\quad \mathrm{Var}\,v=4\,\mathrm{Var}\,u,\quad \mathrm{Cov}=2\,\mathrm{Var}\,u .$$

A uniform field stays uniform under `--smooth` and `--lowpass` on a wet domain, so the answer must not change. Land stays out of the result.

In [ ]:
def domain(name, ny=16, nx=20, nz=3):
    p = os.path.join(TMP, name)
    with netCDF4.Dataset(p, "w") as nc:
        for d, s in (("t", 1), ("z", nz), ("y", ny), ("x", nx)): nc.createDimension(d, s)
        nc.createVariable("nav_lat", "f4", ("y", "x"))[...] = np.linspace(-40, 40, ny)[:, None]*np.ones((ny, nx))
        nc.createVariable("nav_lon", "f4", ("y", "x"))[...] = np.linspace(-170, 170, nx)[None]*np.ones((ny, nx))
        nc.createVariable("top_level", "i4", ("t", "y", "x"))[...] = np.ones((1, ny, nx), int)
        nc.createVariable("bottom_level", "i4", ("t", "y", "x"))[...] = nz*np.ones((1, ny, nx), int)
    return p
def fake(name, gen, ny=60, nx=90, nz=3, **mesh):
    "A dataset module: gen(var, k) gives the (nz, ny, nx) field of snapshot k."
    m = types.ModuleType(f"datakit.{name}")
    gf = os.path.join(TMP, f"{name}_coord.nc")
    with netCDF4.Dataset(gf, "w") as nc:
        nc.createDimension("y", ny); nc.createDimension("x", nx)
        nc.createVariable("nav_lat", "f4", ("y", "x"))[...] = np.linspace(-50, 50, ny)[:, None]*np.ones((ny, nx))
        nc.createVariable("nav_lon", "f4", ("y", "x"))[...] = np.linspace(-179, 179, nx)[None]*np.ones((ny, nx))
    mesh.setdefault("nav_lat", np.linspace(-50, 50, ny)[:, None]*np.ones((ny, nx))); mesh.setdefault("nav_lon", np.linspace(-179, 179, nx)[None]*np.ones((ny, nx)))
    m.files = lambda var, *a, **k: [gf]; m.nsnap = 12
    def stream(*vs, start=None, end=None, level=0, box=None, **k):
        for t in range(m.nsnap):
            fs = [gen(v, t+1) for v in vs]; yield (f"2000-{t+1:02d}", *[f if level is None else f[level] for f in fs])
    m.stream = stream
    m.mesh = lambda *n: mesh[n[0]] if len(n) == 1 else [mesh[x] for x in n]
    sys.modules[f"datakit.{name}"] = m; return m
from noisekit.pipelines import variance as Va, runvar_dmd as Rv, w_modes as Wm, v4_bcbts as V4
dom = domain("dom.nc")
def g_var(v, k):
    f = .1*k*np.ones((3, 60, 90), dtype=np.float32)*(2 if v == "vomecrty" else 1); f[:, 28:32, 40:46] = np.nan; return f
fv = fake("fakev", g_var)
n = fv.nsnap; vu = .01*(n*n-1)/12
with test("noisekit.pipelines.variance.run"):
    for lab, kw in (("plain", {}), ("smooth", dict(smooth=2.)), ("lowpass", dict(lowpass=3.))):
        out = os.path.join(TMP, f"v_{lab}.nc")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore"); Va.run(out, "fakev", level=0, domain=dom, map_file=os.path.join(TMP, "vmap.nc"), **kw)
        with netCDF4.Dataset(out) as nc:
            j = (0, 2)
            if lab == "smooth":
                print("   note: with --smooth, land is filled with 0 before filtering, so only points far from land are compared")
            ok(f"{lab}: mean u", nc["u_tavg"][:][j] - .1*(n+1)/2, 1e-3); ok(f"{lab}: mean v", nc["v_tavg"][:][j] - .1*(n+1), 1e-3)
            ok(f"{lab}: var u", rerr(nc["u_tvar"][:][j], vu), 1e-4); ok(f"{lab}: var v", rerr(nc["v_tvar"][:][j], 4*vu), 1e-4); ok(f"{lab}: cov", rerr(nc["uv_tvcov"][:][j], 2*vu), 1e-4)

### `runvar_dmd`: `_months_before`, `window_stat`, `decompose`, `run` (D1, D2)

For $u_k=k$ and a window of $w=3$ snapshots, the running variance is the constant $(w^2-1)/12=2/3$, and $\sqrt{2/3}$ for `std`.

For $u_k=a\cos\omega k$, the running variance over 3 points is, exactly,

$$\mathrm{Var}_k=\frac{a^2}{2}\Big[1-\frac{D^2}{9}\Big]+\frac{a^2}{2}\Big[\frac{D_2}{3}-\frac{D^2}{9}\Big]\cos\big(2\omega(k-1)\big),\quad D=1+2\cos\omega,\;D_2=1+2\cos2\omega,$$

a mean plus one harmonic at $2\omega$. So DMD of its anomalies must find exactly $\lambda=e^{\pm2i\omega}$, in both the exact and the kernel variants, and `run` must write the mean and the frequency $2\omega/\Delta t$.

In [ ]:
with test("noisekit.pipelines.runvar_dmd._months_before"):
    yes("across a year", Rv._months_before("2016-01", 2) == "2015-11"); yes("same year", Rv._months_before("2016-05", 1) == "2016-04")
with test("noisekit.pipelines.runvar_dmd.window_stat"):
    fr = fake("faker", lambda v, k: k*np.ones((3, 20, 30)), ny=20, nx=30); fr.nsnap = 8
    s = Rv.window_stat(fr, "vozocrtx", "2000-03", "2000-08", 3, 0, np.ones((20, 30)), "var", 0); yes("window count", s.shape[0] == 6); ok("var = 2/3", s - 2/3, 1e-12)
    s = Rv.window_stat(fr, "vozocrtx", "2000-03", "2000-08", 3, 0, np.ones((20, 30)), "std", 0); ok("std", s - np.sqrt(2/3), 1e-12)
om = 2*np.pi/9; a_ = 2.; D1_ = 1+2*np.cos(om); D2_ = 1+2*np.cos(2*om)
vmean = a_**2/2*(1 - D1_**2/9); vamp = a_**2/2*(D2_/3 - D1_**2/9)
with test("noisekit.pipelines.runvar_dmd.decompose"):
    ka = np.arange(2, 40); xv = vmean + vamp*np.cos(2*om*(ka-1)); direct = np.array([np.var(a_*np.cos(om*np.arange(k-2, k+1))) for k in ka])
    ok("closed form of the running variance", xv - direct, 1e-12)
    Xa = np.outer(np.sin(np.linspace(0, 3, 50)), xv - xv.mean())
    Xa = np.outer(np.sin(np.linspace(0, 3, 50)), vamp*np.cos(2*om*(ka-1))); Xa = np.vstack([Xa, np.outer(np.cos(np.linspace(0, 3, 50)), vamp*np.sin(2*om*(ka-1)))])
    mds, eig, ct = Rv.decompose(Xa, 2, None, 5.); ok("exact DMD finds exp(2 i omega)", match(np.exp([2j*om, -2j*om]), eig), 1e-10); ok("continuous", match(np.array([2j*om, -2j*om])/5., ct), 1e-10)
    mds, eig, ct = Rv.decompose(Xa, 2, "linear", 5.); ok("kernel DMD finds exp(2 i omega)", match(np.exp([2j*om, -2j*om]), eig), 1e-8)
with test("noisekit.pipelines.runvar_dmd.run"):
    def g_r(v, k):
        # the phase varies with x, so the running variance has rank 2
        return a_*np.cos(om*k + 2*np.pi*np.arange(60)/60)[None, None]*np.ones((3, 40, 60), dtype=np.float64)
    fr2 = fake("faker2", g_r, ny=40, nx=60, tmask=np.ones((3, 40, 60))); fr2.nsnap = 47
    dom2 = domain("dom2.nc")
    out = os.path.join(TMP, "rv.nc")
    Rv.run(out, "2000-03", "2000-47", "faker2", window=3, rank=2, sigma=0, map_file=os.path.join(TMP, "rmap.nc"), domain=dom2)
    with netCDF4.Dataset(out) as nc:
        um = nc["uavg"][:]; print("   modes", nc["omega"].shape, "omega", nc["omega"][:])
        ok("mean running variance", rerr(um[0, 8, 10], vmean), 1e-6); ok("written omega x dt", (match(np.array([2*om/Rv.DT_MONTH]), np.abs(nc["omega"][:])))*Rv.DT_MONTH, 1e-6)

### `w_modes`: `w_from_divergence`, `snapshot_w`, `run` (W1)

Continuity with $k$ counted downward: $w_k=w_{k+1}-e_3\,\nabla\!\cdot\mathbf u_k$, $w_{\text{bottom}}=0$. For a constant divergence $\delta$ and thickness $e_3$, with the last level dry as on every NEMO grid,

$$w_k=-\delta\,e_3\,(n_{\text{wet}}-k).$$

`snapshot_w` must get $\delta$ from the velocities: $u=\delta_1x$, $v=\delta_2y$, $\delta=\delta_1+\delta_2$, with the correct sign of $\partial_yv$, which was the bug in the old notebook. `run` writes the mean of $w$ and modes that rebuild its oscillation.

In [ ]:
with test("orcakit.grid_c.w_from_divergence"):
    dl = 2e-6; e3 = 10*np.ones((5, 2, 2)); tm = np.ones((5, 2, 2)); tm[-1] = 0; w = grid_c.w_from_divergence(dl*tm, e3, tm)
    ok("w_k = -delta e3 (4-k)", w[:, 0, 0] + dl*10*(4-np.arange(5)), 1e-18)
    ok("the real meshes have a dry last level, which this relies on", [glorys04.mesh("tmask")[-1].sum(), cmems.mesh("tmask")[-1].sum()], 0)
nyw, nxw = 24, 30; dxw, dyw = 1e4, 2e4
def wgrid(nz=4):
    O = np.ones((nyw, nxw)); tm = np.ones((nz, nyw, nxw)); tm[-1] = 0
    return dict(e1t=dxw*O, e2t=dyw*O, e1v=dxw*O, e2u=dyw*O, e3t=10*tm*0+10., tmask=tm)
with test("noisekit.pipelines.w_modes.snapshot_w"):
    d1, d2 = 1e-6, 3e-6; jj, ii = np.mgrid[:nyw, :nxw].astype(float); u = d1*(ii+.5)*dxw; v = d2*(jj+.5)*dyw
    m_ = wgrid(); w = Wm.snapshot_w(np.stack([u]*4), np.stack([v]*4), m_)
    ok("surface w = -(d1+d2) e3 n_wet", w[0, 5:-5, 5:-5] + (d1+d2)*10*3, 1e-16)
with test("noisekit.pipelines.w_modes.run"):
    mt = wgrid(); nzw = 4; jj, ii = np.mgrid[:nyw, :nxw].astype(float)
    kx_, ky_ = 2*np.pi*3/(nxw*dxw), 2*np.pi*2/(nyw*dyw); pw_ = 2*np.pi/9
    def g_w(v, k):
        # uniform divergence plus two oscillating patterns in quadrature
        return np.stack([1e-6*(ii+.5)*dxw + .5e-6*np.cos(pw_*k)*np.sin(kx_*(ii+.5)*dxw)/kx_ if v == "vozocrtx" else 2e-6*(jj+.5)*dyw + .5e-6*np.sin(pw_*k)*np.sin(ky_*(jj+.5)*dyw)/ky_]*nzw)
    fw = fake("fakew", g_w, ny=nyw, nx=nxw, e1t=mt["e1t"], e2t=mt["e2t"], e1v=mt["e1v"], e2u=mt["e2u"], e3t_0=10*np.ones(nzw), tmask=mt["tmask"]); fw.nsnap = 36
    dom3 = domain("dom3.nc", ny=10, nx=12, nz=nzw); out = os.path.join(TMP, "wm.nc")
    Wm.run(out, "2000-01", "2000-36", "fakew", rank=2, tol=.05, map_file=os.path.join(TMP, "wmap.nc"), domain=dom3)
    with netCDF4.Dataset(out) as nc:
        ok("mean w at the surface = -(1e-6+2e-6) e3 n_wet", rerr(nc["wavg"][0, 5, 6], -3e-6*10*3), 2e-2); yes("modes written", nc["omega"].size > 0)
        print("   omega x dt / (2 pi / 9) =", np.abs(nc["omega"][:])*Wm.DT_MONTH/(2*np.pi/9))
        ok("mode frequency", match(np.array([2*np.pi/9/Wm.DT_MONTH]), np.abs(nc["omega"][:]))*Wm.DT_MONTH, 1e-6)

### `v4_bcbts`: `accumulate`, `lowpass`, `to_orca1`, `to_tlu` (V4)

With $u_t(z)=B_t+c(z)$, $\sum c\,e_3=0$ and $B_t=t$: the barotropic part has mean $\bar t$ and variance $\mathrm{Var}\,t$, the baroclinic part has mean $c(z)$ and **zero** variance. A horizontal low-pass of a uniform field changes nothing. `to_tlu` must copy only the baroclinic half under the names NEMO reads, with `uv_tvcov` zero.

In [ ]:
nz4 = 3; ny4, nx4 = 60, 90; cz = np.array([1., -1., 1.])
e30 = np.array([1., 2., 1.]); eps = np.ones((ny4, nx4))
tm4 = np.ones((nz4, ny4, nx4), np.float32)
def g4(v, k): return (k + cz[:, None, None]*(2 if v == "vomecrty" else 1))*np.ones((nz4, ny4, nx4), np.float32)
f4 = fake("fake4", g4, ny=ny4, nx=nx4, e3t_0=e30, e3t_ps=eps, tmask=tm4, umask=tm4, vmask=tm4); f4.nsnap = 11; n4 = f4.nsnap
V4.glorys04 = f4
with test("noisekit.pipelines.v4_bcbts.accumulate"):
    st4, um4, vm4 = V4.accumulate(progress_every=1000)
    ok("barotropic mean", st4["ubt"][0][5, 5] - (n4+1)/2, 1e-4); ok("barotropic variance", rerr(st4["ubt"][1][5, 5], (n4*n4-1)/12), 1e-4)
    ok("baroclinic mean = c(z)", st4["ubc"][0][:, 5, 5] - cz, 1e-4); ok("baroclinic variance = 0", st4["ubc"][1][:, 5, 5], 1e-4); ok("v baroclinic mean", st4["vbc"][0][:, 5, 5] - 2*cz, 1e-4)
with test("noisekit.pipelines.v4_bcbts.lowpass"):
    lp = V4.lowpass(st4, um4, vm4, sigma=3.); ok("uniform barotropic mean unchanged", lp["ubt_mean"][30, 45] - (n4+1)/2, 1e-3); ok("baroclinic mean unchanged", lp["ubc_mean"][:, 30, 45] - cz, 1e-3)
with test("noisekit.pipelines.v4_bcbts.to_orca1", "noisekit.pipelines.v4_bcbts.to_tlu"):
    dm4 = domain("dom4.nc"); V4.DOMAIN_ORCA1 = dm4; c4 = V4.to_orca1(lp, os.path.join(TMP, "m4.nc"), domain=dm4); ok("sampled", c4["ubt_mean"][4, 4] - (n4+1)/2, 1e-3)
    stf = os.path.join(TMP, "s4.nc"); tf = os.path.join(TMP, "t4.nc"); onc.write_fields(dm4, stf, c4); V4.to_tlu(stf, tf)
    with netCDF4.Dataset(tf) as nc:
        ok("u_tavg = baroclinic mean", nc["u_tavg"][:, 4, 4] - cz, 1e-3); ok("v_tavg", nc["v_tavg"][:, 4, 4] - 2*cz, 1e-3); ok("u_tvar", nc["u_tvar"][:, 4, 4].filled(0), 1e-3); ok("uv_tvcov = 0", nc["uv_tvcov"][:].filled(0), 0)

### V5 production scripts (`build_noise_sigma`, `build_noise_lscale`, `build_noise_BcResid_stats`, `build_baroclinic_daily`)

Everything runs here in one process on small temporary files.

- `_masked_gaussian_padded`, `highpass_tile`: the high-pass transfer function is $1-e^{-k^2\sigma^2/2}$, and $\text{raw}=\text{low}+\text{high}$.
- `accumulate_lag_products`, `efold_map`: for $u'=\cos(kx+\theta_t)$ with evenly spaced phases, $\rho_x(\ell)=\cos k\ell$. For $\rho=e^{-\ell/L}$ the e-folding is $L$.
- `write_noise_sigma`, `write_noise_lscale`, `to_orca1`: a constant field stays constant through the coarse-grid binning. $r=L_x/L_y$ holds, with its clamp.
- `bc_residual`: a field with no small scales has a residual of nearly zero; uniform and baroclinic-free gives exactly 0.
- `baroclinic_field`, `clone_with`: $u_{bc}=u-\sum ue_3/\sum e_3$; land is NaN; all other variables are copied unchanged.

In [ ]:
from noisekit.pipelines import build_noise_sigma as Bs, build_noise_lscale as Bl
ny5, nx5 = 60, 120; nz5 = 2; dd = os.path.join(TMP, "bc"); os.makedirs(os.path.join(dd, "vozocrtx"), exist_ok=True); os.makedirs(os.path.join(dd, "vomecrty"), exist_ok=True)
k5 = 2*np.pi*8/(nx5-2); xx = np.arange(nx5)
def daily(path, var, f):
    with netCDF4.Dataset(path, "w") as nc:
        for d, s in (("z", nz5), ("y", ny5), ("x", nx5)): nc.createDimension(d, s)
        nc.createVariable(var, "f4", ("z", "y", "x"), fill_value=1e20)[...] = f
        nc.createVariable("nav_lat", "f4", ("y", "x"))[...] = np.linspace(-50, 50, ny5)[:, None]*np.ones((ny5, nx5))
        nc.createVariable("nav_lon", "f4", ("y", "x"))[...] = np.linspace(-179, 179, nx5)[None]*np.ones((ny5, nx5))
for d_, th in ((1, 0.), (2, 1.)):
    fw_ = np.cos(k5*xx+th)[None, None]*np.ones((nz5, ny5, nx5))
    daily(f"{dd}/vozocrtx/vozocrtx_glorys04_2015-01-0{d_}.nc", "vozocrtx", fw_); daily(f"{dd}/vomecrty/vomecrty_glorys04_2015-01-0{d_}.nc", "vomecrty", 2*fw_)
Bs.database_dir = dd
dom5 = domain("dom5.nc", ny=10, nx=12, nz=nz5); Bs.orca1_domcfg = dom5; Bl.orca1_domcfg = dom5
sig5 = 3.; hw5 = 12; dec1 = mo.Decomp.serial_decomp(ny5, nx5)
G5 = np.exp(-k5**2*sig5**2/2); WV = np.cos(k5*xx)[None, None]*np.ones((nz5, ny5+2*hw5, nx5+2*hw5)) if False else None
with test("noisekit.pipelines.build_noise_sigma.discover_files", "noisekit.pipelines.build_noise_sigma._grid_size"):
    uf, vf = Bs.discover_files(); yes("two pairs", len(uf) == 2 and len(vf) == 2); yes("limit", len(Bs.discover_files(limit=1)[0]) == 1); yes("year filter", len(Bs.discover_files(years=[2014])[0]) == 0); ok("size", np.array(Bs._grid_size(uf[0])) - [ny5, nx5], 0)
with test("noisekit.pipelines.build_noise_sigma._masked_gaussian_padded", "noisekit.pipelines.build_noise_sigma.highpass_tile"):
    a = np.cos(k5*(np.arange(nx5+2*hw5)-hw5))[None, None]*np.ones((nz5, ny5+2*hw5, nx5+2*hw5)); w = np.ones_like(a)
    lp5 = Bs._masked_gaussian_padded(a, w, sig5); ok("low-pass transfer", rerr(lp5[:, 30:-30, hw5:-hw5], G5*a[:, 30:-30, hw5:-hw5]), 2e-3)
    hp5 = Bs.highpass_tile(a, w, sig5, hw5); yes("interior shape", hp5.shape == (nz5, ny5, nx5)); ok("high-pass transfer", rerr(hp5[:, 20:-20], (1-G5)*a[:, 20+hw5:-20-hw5, hw5:-hw5]), 2e-3)
    yes("margin kept", Bs.highpass_tile(a, w, sig5, hw5, margin=4).shape == (nz5, ny5+8, nx5+8))
with test("noisekit.pipelines.build_noise_sigma.exchange_lowpass", "noisekit.pipelines.build_noise_sigma.process_one_day", "noisekit.pipelines.build_noise_sigma.reference_highpass", "noisekit.pipelines.build_noise_sigma.reference_lowpass"):
    fld = np.cos(k5*xx)[None, None]*np.ones((nz5, ny5, nx5), np.float32); wt = np.ones_like(fld)
    ok("exchange_lowpass transfer", rerr(Bs.exchange_lowpass(dec1, fld, wt, sig5, False)[:, 20:-20], G5*fld[:, 20:-20]), 2e-3)
    hu, hv = Bs.process_one_day(dec1, uf[0], vf[0], sig5, hw5); ok("process_one_day u", rerr(hu[:, 20:-20], (1-G5)*fld[:, 20:-20]), 2e-3); ok("process_one_day v = 2u", rerr(hv[:, 20:-20], 2*(1-G5)*fld[:, 20:-20]), 2e-3)
    ok("reference_highpass", rerr(Bs.reference_highpass(uf[0], "vozocrtx", sig5, False)[:, 20:-20], (1-G5)*fld[:, 20:-20]), 2e-3)
    ok("reference_lowpass", rerr(Bs.reference_lowpass(fld, wt, sig5, False)[:, 20:-20], G5*fld[:, 20:-20]), 2e-3)
with test("noisekit.pipelines.build_noise_sigma.check_tiling"):
    r = Bs.check_tiling(dec1, uf[0], vf[0], sig5, hw5, sig5); yes("ran in one process", True)
with test("noisekit.pipelines.build_noise_sigma.write_noise_sigma"):
    Bs.output_file = os.path.join(TMP, "ns.nc"); la5 = np.linspace(-50, 50, ny5)[:, None]*np.ones((ny5, nx5)); lo5 = np.linspace(-179, 179, nx5)[None]*np.ones((ny5, nx5)); C5 = np.ones((nz5, ny5, nx5))
    Bs.write_noise_sigma(la5, lo5, C5, C5, 2*C5, 3*C5, 4*C5, 5*C5, 6*C5)
    with netCDF4.Dataset(Bs.output_file) as nc:
        ok("constants survive", [nc["u_tavg"][0, 5, 5]-2, nc["v_tavg"][0, 5, 5]-3, nc["u_tvar"][0, 5, 5]-4, nc["v_tvar"][0, 5, 5]-5, nc["uv_tvcov"][0, 5, 5]-6], 1e-5)
with test("noisekit.pipelines.build_noise_lscale.accumulate_lag_products", "noisekit.pipelines.build_noise_lscale.efold_map"):
    km = 5; nt5 = 16; ny6, nx6 = 8, 64; k6 = 2*np.pi*4/nx6; Px, Py = np.zeros((km, ny6, nx6)), np.zeros((km, ny6, nx6)); P0 = np.zeros((ny6+2*km, nx6+2*km))
    for t in range(nt5):
        th = 2*np.pi*t/nt5; h = np.cos(k6*(np.arange(nx6+2*km)-km)+th)[None, None]*np.ones((1, ny6+2*km, nx6+2*km)); Bl.accumulate_lag_products(h, 0*h, km, Px, Py, P0)
    ok("rho_x = cos(k l)", Px[:, 3, 10]/P0[3+km, 10+km] - np.cos(k6*np.arange(1, km+1)), 1e-6); ok("rho_y = 1", Py[:, 3, 10]/P0[3+km, 10+km] - 1, 1e-6)
    L0 = 5.5; Lm, cm = Bl.efold_map(np.exp(-np.arange(1, 21)/L0)[:, None, None]*np.ones((20, 2, 2)), 20); ok("e-fold", Lm - L0, 1.1/(8*L0)); yes("not censored", not cm.any())
with test("noisekit.pipelines.build_noise_lscale.write_noise_lscale"):
    mm = os.path.join(TMP, "mm.nc")
    with netCDF4.Dataset(mm, "w") as nc:
        for d, s in (("t", 1), ("y", ny5), ("x", nx5)): nc.createDimension(d, s)
        nc.createVariable("e1t", "f4", ("t", "y", "x"))[...] = 1e4*np.ones((1, ny5, nx5)); nc.createVariable("e2t", "f4", ("t", "y", "x"))[...] = 2e4*np.ones((1, ny5, nx5))
        nc.createVariable("nav_lat", "f4", ("y", "x"))[...] = la5; nc.createVariable("nav_lon", "f4", ("y", "x"))[...] = lo5
    Bl.mesh_mask_file = mm; Bl.output_file = os.path.join(TMP, "nl.nc"); Bl.write_noise_lscale(3*np.ones((ny5, nx5)), 2*np.ones((ny5, nx5)), np.ones((1, ny5, nx5)))
    with netCDF4.Dataset(Bl.output_file) as nc:
        ok("Lx", nc["Lx"][5, 5] - 3e4, 1e-2); ok("Ly", nc["Ly"][5, 5] - 4e4, 1e-2); ok("r = Lx/Ly", nc["r"][5, 5] - .75, 1e-5)
with test("noisekit.pipelines.build_noise_lscale.check_tiling"):
    rho = np.ones((3, 8, 8)); Bl.check_tiling(mo.Decomp.serial_decomp(8, 8), .5*rho, .5*rho, 3); yes("ran in one process", True)
ORCA_NS = dict(np=np, netcdf=netCDF4, os=os, re=__import__("re"), glob=glob, orca=orca, filters=filters, acc_dtype=np.float32, sigma_cg=2.)
with test("noisekit.pipelines.build_noise_BcResid_stats.discover_months", "noisekit.pipelines.build_noise_BcResid_stats.bc_residual", "noisekit.pipelines.build_noise_BcResid_stats.to_orca1"):
    ns = load_funcs("noisekit/pipelines/build_noise_BcResid_stats.py", ("discover_months", "bc_residual", "to_orca1"), **ORCA_NS)
    ok("months", len(ns["discover_months"](f"{dd}/vozocrtx", "vozocrtx")) - 1, 0)
    ny7, nx7 = 40, 60; la7 = np.linspace(-50, 50, ny7)[:, None]*np.ones((ny7, nx7)); lo7 = np.linspace(-179, 179, nx7)[None]*np.ones((ny7, nx7)); lc = np.linspace(-40, 40, 8)[:, None]*np.ones((8, 10)); oc = np.linspace(-170, 170, 10)[None]*np.ones((8, 10))
    fl_ = orca.assign_to_coarse_grid(la7, lo7, lc, oc, np.ones((8, 10), bool))
    ns.update(coarse_flat_idx=fl_, flat=fl_.ravel(), n_coarse=lc.size, nz=2, lat_orca1=lc, mask_orca1=np.ones((8, 10), bool))
    pth = os.path.join(TMP, "bcres.nc"); cc = np.array([1., -1.])[:, None, None]*np.ones((2, ny7, nx7))
    with netCDF4.Dataset(pth, "w") as nc:
        nc.createDimension("z", 2); nc.createDimension("y", ny7); nc.createDimension("x", nx7); nc.createVariable("q", "f4", ("z", "y", "x"))[...] = 3 + cc
    wm7 = np.ones((2, ny7, nx7), np.float32); sm7 = partial(orca.smooth_2D, mask=None)
    res = ns["bc_residual"](pth, "q", np.ones((2, ny7, nx7), np.float32), wm7, sm7, sm7); ok("depth-uniform plus baroclinic constant has no residual", res, 1e-4)
    ok("to_orca1 keeps a constant", ns["to_orca1"](2*wm7, wm7)[:, 3, 3] - 2, 1e-5)
with test("noisekit.pipelines.build_baroclinic_daily.log", "noisekit.pipelines.build_baroclinic_daily.discover_pairs", "noisekit.pipelines.build_baroclinic_daily.build_metric", "noisekit.pipelines.build_baroclinic_daily.baroclinic_field", "noisekit.pipelines.build_baroclinic_daily.clone_with"):
    ns = load_funcs("noisekit/pipelines/build_baroclinic_daily.py", ("log", "discover_pairs", "build_metric", "baroclinic_field", "clone_with"), np=np, netcdf=netCDF4, os=os, glob=glob, time=time, sys=sys, orca=orca, acc_dtype=np.float32, rank=0, size=1, database_dir=dd, udate_re=__import__("re").compile(r"vozocrtx_glorys04_(\d{4})-(\d{2})-(\d{2})\.nc$"))
    ns["log"]("hello"); yes("pairs", len(ns["discover_pairs"]("all")) == 2); yes("year filter", len(ns["discover_pairs"]("2014")) == 0)
    mm2 = os.path.join(TMP, "mm2.nc")
    with netCDF4.Dataset(mm2, "w") as nc:
        for d, s in (("t", 1), ("z", 2), ("y", 5), ("x", 4)): nc.createDimension(d, s)
        nc.createVariable("e3t_0", "f4", ("t", "z"))[...] = [[1., 3.]]; nc.createVariable("e3t_ps", "f4", ("t", "y", "x"))[...] = 3*np.ones((1, 5, 4))
        for m_ in ("tmask", "umask", "vmask"): nc.createVariable(m_, "f4", ("t", "z", "y", "x"))[...] = np.ones((1, 2, 5, 4))
    ns["mesh_mask_file"] = mm2; e3u5, e3v5, um5, vm5 = ns["build_metric"](); ok("e3u", e3u5[:, 1, 1] - [1, 3], 1e-6)
    src = os.path.join(TMP, "src.nc")
    with netCDF4.Dataset(src, "w") as nc:
        nc.setncattr("title", "T"); nc.createDimension("z", 2); nc.createDimension("y", 5); nc.createDimension("x", 4)
        v = nc.createVariable("q", "f4", ("z", "y", "x"), fill_value=1e20, zlib=True); v[...] = np.array([4., 8.])[:, None, None]*np.ones((2, 5, 4)); v.units = "m/s"; nc.createVariable("keep", "f4", ("y",))[:] = [1, 2, 3, 4, 5]
    mk5 = np.ones((2, 5, 4), bool); mk5[:, 0, 0] = False
    bc = ns["baroclinic_field"](src, "q", e3u5, mk5); ok("u - sum(u e3)/sum(e3): mean 7", bc[:, 1, 1] - [-3, 1], 1e-5); yes("land NaN", np.isnan(bc[:, 0, 0]).all())
    dst = os.path.join(TMP, "dst.nc"); ns["clone_with"](src, dst, "q", bc)
    with netCDF4.Dataset(dst) as nc:
        ok("field replaced", nc["q"][:, 1, 1] - [-3, 1], 1e-5); ok("other variable kept", nc["keep"][:] - [1, 2, 3, 4, 5], 0); yes("attributes kept", nc.title == "T" and nc["q"].units == "m/s")

## 10. MPI layer (`orcakit.mpi_orca`, `orcakit.mpi_filter`)

These run under `mpirun -n 4` in a subprocess and also in one process, on a label field $f_{j,i}=1000j+i$ in a small file. Each ghost cell has a known value:

- east and west: wrap, $f_{j,(i\pm n)\bmod n_x}$
- south: interior rows reflected, as in `smooth_2D`
- north: $f_{n_y-2-p,\;n_x-1-i}$, and its negative with `is_meridional`

Tiled results must equal one-process results exactly. The Gaussian `Tile.smooth` must attenuate a wave by $e^{-k^2\sigma^2/2}$. Tiling must not change a result: this is an identity, not a comparison with old output. `ok` here records the largest error over all ranks.

In [ ]:
mpi_code = textwrap.dedent('''
import json, os, sys
import numpy as np, netCDF4
sys.path.insert(0, sys.argv[1]); TMP = sys.argv[2]
from mpi4py import MPI
from orcakit import mpi_orca as mo, mpi_filter as mf
comm = MPI.COMM_WORLD; rank = comm.Get_rank(); R = []
def ok(d, err, tol=0.0):
    R.append((d, float(np.max(np.abs(err))) if np.size(err) else 0.0, tol))
nz, ny, nx = 2, 48, 64; G = 1000.*np.arange(ny)[:, None] + np.arange(nx); F = np.stack([G, G+1e5]).astype(np.float32)
p = os.path.join(TMP, "lab.nc"); pw = os.path.join(TMP, "wave.nc"); k = 2*np.pi*4/(nx-2); W = (np.cos(k*np.arange(nx))[None, None]*np.ones((nz, ny, nx))).astype(np.float32)
if rank == 0:
    for q, v in ((p, F), (pw, W)):
        with netCDF4.Dataset(q, "w") as nc:
            nc.createDimension("z", nz); nc.createDimension("y", ny); nc.createDimension("x", nx); nc.createVariable("f", "f4", ("z", "y", "x"))[...] = v
comm.Barrier()
dec = mo.Decomp.for_file(p, "f"); ok("Decomp ranks = %d" % comm.Get_size(), dec.nranks - comm.Get_size())
hw = 3; jsl, isl = dec.slices(); ok("slices pick the tile", F[(0, jsl, isl)] - F[0][jsl, isl])
sub, wet = mo.read_slab(dec, p, "f"); ok("read_slab", sub - F[:, jsl, isl]); ok("wet = 1", wet - 1)
ok("read_slab level", mo.read_slab(dec, p, "f", level=1)[0] - F[1:2, jsl, isl])
a = mo.add_halo(dec, sub, hw); ok("strip_halo inverse of add_halo", mo.strip_halo(a, hw) - sub)
jj = np.arange(dec.j0-hw, dec.j0+dec.jl+hw); ii = np.arange(dec.i0-hw, dec.i0+dec.il+hw)
E = np.empty((len(jj), len(ii)))
for b, j in enumerate(jj):
    for c, i in enumerate(ii):
        E[b, c] = G[-1-j if j < 0 else (ny-4-(j-ny) if j >= ny else j), (nx-i) % (nx-2) if j >= ny else (i % (nx-2) if i < 0 or i >= nx else i)]
south = dec.j0 == 0; north = dec.j0 + dec.jl == ny
ok("add_halo west and east and interior rows", (a[0] - E)[hw:hw+dec.jl])
if south: ok("south reflects the interior rows", (a[0] - E)[:hw, hw:-hw])
if north: ok("north fold ghost, interior columns", (a[0] - E)[-hw:, hw:-hw]); ok("north fold ghost, the corners too", (a[0] - E)[-hw:], 0.0)
if not south: ok("y neighbour exchange south", (a[0] - E)[:hw])
if not north: ok("y neighbour exchange north", (a[0] - E)[-hw:])
am = mo.add_halo(dec, sub, hw, is_meridional=True)
if north: ok("is_meridional negates the fold", (am[0] + E)[-hw:, hw:-hw])
b, w = mo.read_slab_with_halo(dec, p, "f", hw); ok("read_slab_with_halo = labels", (b[0] - E)[hw:]); ok("halo wet", w - 1)
ok("read_slab_with_halo level", mo.read_slab_with_halo(dec, p, "f", hw, level=1)[0].shape[0] - 1)
g = mo.gather(dec, sub)
if rank == 0: ok("gather", g - F)
n = mo.Decomp.auto(ny, nx); ok("auto agrees", n.jl - dec.jl)
ok("allreduce_sum", dec.allreduce_sum(np.ones(3)) - comm.Get_size()); dec.barrier(); ok("edges", [dec.is_north_edge != north, dec.is_south_edge != south])
ok("_split balanced", np.array(mo._split(10, 3)) - [[0, 4], [4, 7], [7, 10]])
ok("mpi_orca.halo = orca.halo", mo.halo(G, np.array([1,2,1])) - __import__("orcakit.orca", fromlist=["x"]).halo(G, np.array([1,2,1])))
NOISE = np.random.default_rng(0).standard_normal((nz, ny, nx)).astype(np.float32)
pn = os.path.join(TMP, "noise.nc")
if rank == 0:
    with netCDF4.Dataset(pn, "w") as nc:
        nc.createDimension("z", nz); nc.createDimension("y", ny); nc.createDimension("x", nx); nc.createVariable("f", "f4", ("z", "y", "x"))[...] = NOISE
comm.Barrier()
O_ = __import__("orcakit.orca", fromlist=["x"])
for P_, sgn in (("T", False), ("U", True), ("V", True), ("F", False)):
    sub_ = NOISE[:, jsl, isl]; ax = mo.add_halo(dec, sub_, hw, is_meridional=sgn, point=P_); ar, _ = mo.read_slab_with_halo(dec, pn, "f", hw, is_meridional=sgn, point=P_)
    ok(f"{P_}: add_halo = read_slab_with_halo, whole padded tile", ax - ar, 0.0)
    hs = O_.halo(NOISE, np.array([hw, hw, hw]), sgn, P_); south_r = hs[:, hw-1::-1] if False else None
    e_ = np.empty_like(ax[0]);
    for b_, j_ in enumerate(jj):
        rj = -1-j_ if j_ < 0 else j_
        e_[b_] = hs[0, rj, ii + hw]
    ok(f"{P_}: tile ghosts = orca.halo, whole padded tile", ax[0] - e_, 1e-6)
ok("mpi_orca.dehalo", mo.dehalo(mo.halo(G, np.array([1,2,1])), np.array([1,2,1])) - G)
# mpi_filter
s = 2.5; G5 = np.exp(-k*k*s*s/2)
t = mf.Tile.from_mpi(comm, pw, "f", s, False) if comm.Get_size() > 1 else mf.Tile.from_whole(pw, "f", s, False)
sm = t.smooth(s); full = t.gather_to_root(sm)
if rank == 0:
    ok("Tile.smooth transfer", (full - G5*W)[:, 12:-12], 3e-3)
    ref = mf.reference_smooth(pw, "f", s, False); sI = 3.0; tI = mf.Tile.from_mpi(comm, pw, "f", sI, False) if comm.Get_size() > 1 else mf.Tile.from_whole(pw, "f", sI, False)
if rank != 0:
    sI = 3.0; tI = mf.Tile.from_mpi(comm, pw, "f", sI, False) if comm.Get_size() > 1 else mf.Tile.from_whole(pw, "f", sI, False)
fI = tI.gather_to_root(tI.smooth(sI))
if rank == 0:
    refI = mf.reference_smooth(pw, "f", sI, False); ok("Tile equals reference_smooth, smooth wave", fI-refI, 2e-5)
for P_, sgn in (("T", False), ("U", True), ("V", True), ("F", False)):
    for s_ in (3.0, 2.5):
        tn = mf.Tile.from_mpi(comm, pn, "f", s_, sgn, P_) if comm.Get_size() > 1 else mf.Tile.from_whole(pn, "f", s_, sgn, P_)
        fn = tn.gather_to_root(tn.smooth(s_))
        if rank == 0: ok(f"{P_}, sigma {s_}: Tile = reference_smooth on white noise, everywhere", fn-mf.reference_smooth(pn, "f", s_, sgn, P_), 2e-5)
ok("mpi_filter._split", np.array(mf._split(10, 3)) - [[0, 4], [4, 7], [7, 10]])
ok("mpi_filter._read_slab", mf._read_slab(p, "f")[0] - F)
if not isinstance(mf.default_path("vozocrtx", "2015-01-01"), str): ok("default_path", 1)
A_ = comm.gather(R, root=0)
if rank == 0:
    best = {}
    for part in A_:
        for d, e, tol in part: best[d] = (max(e, best.get(d, (0, 0))[0]), tol)
    print("JSON" + json.dumps([(d, e, tol) for d, (e, tol) in best.items()]))
''')
mpath = os.path.join(TMP, "mpi_case.py"); open(mpath, "w").write(mpi_code)
def run_mpi(n):
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", OMPI_ALLOW_RUN_AS_ROOT="1", OMPI_ALLOW_RUN_AS_ROOT_CONFIRM="1", OMPI_MCA_rmaps_base_oversubscribe="1")
    cmd = ["mpirun", "-n", str(n), sys.executable, mpath, ROOT, TMP] if n > 1 else [sys.executable, mpath, ROOT, TMP]
    p = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=300)
    if p.returncode: raise RuntimeError(p.stderr[-800:])
    return __import__("json").loads([l for l in p.stdout.splitlines() if l.startswith("JSON")][0][4:])
NAMES = ["_split", "halo", "dehalo", "Decomp.__init__", "Decomp.serial_decomp", "Decomp.from_comm", "Decomp.auto", "Decomp.for_file", "Decomp.slices", "Decomp.is_north_edge", "Decomp.is_south_edge", "Decomp.barrier", "Decomp.allreduce_sum", "read_slab", "read_slab_with_halo", "gather", "_north_fold", "exchange_halo", "add_halo", "strip_halo"]
FNAMES = ["default_path", "_split", "_read_slab", "Tile.__init__", "Tile.from_whole", "Tile.from_mpi", "Tile._exchange", "Tile._north_fold", "Tile.smooth", "Tile.gather_to_root", "reference_smooth"]
with test(*[f"orcakit.mpi_orca.{n}" for n in NAMES], *[f"orcakit.mpi_filter.{n}" for n in FNAMES]):
    for nr in (1, 4):
        print(f"   --- {nr} rank(s)")
        for d, e, tol in run_mpi(nr): ok(f"[{nr}] {d}", e, tol)

## Result

### All failures

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS, columns=["routine", "status", "check", "error", "tol"])
print(df.status.value_counts().to_string(), f"\n{df.routine.nunique()} test groups, {len(df)} checks\n")
pd.set_option("display.max_colwidth", 90, "display.width", 200, "display.max_rows", 200)
df[df.status != "PASS"].reset_index(drop=True)

### Coverage

Every function, class method and property in the kits is listed from the source files, then compared with what the tests above declared. Only command-line entry points are left out, because they just call what is tested.

In [ ]:
found = {}
for f in sorted(glob.glob(os.path.join(ROOT, "*kit", "**", "*.py"), recursive=True)):
    mod = os.path.relpath(f, ROOT)[:-3].replace("/", ".")
    if mod.endswith("__init__"): continue
    for n in ast.parse(open(f).read()).body:
        if isinstance(n, ast.FunctionDef): found[f"{mod}.{n.name}"] = mod
        if isinstance(n, ast.ClassDef):
            for m in n.body:
                if isinstance(m, ast.FunctionDef): found[f"{mod}.{n.name}.{m.name}"] = mod
ENTRY = ("main", "parse_args")
missing = [k for k in found if k not in COVERED and k.split(".")[-1] not in ENTRY]
print(f"{len(found)} routines, {len([k for k in found if k.split('.')[-1] in ENTRY])} command-line entry points excluded, {len(missing)} not covered")
print("\n".join(missing))